In [15]:
# ============================================================
# IDPC Reproduction (Binder Demo)
# ------------------------------------------------------------
# Lightweight demonstration of EEG–quantum structural analysis
# from the IDPC study.
#
# This notebook illustrates:
#   • structured agreement
#   • phase synchronization
#   • discrete states
#   • localized switching structure
#
# using a simplified pipeline suitable for browser execution.
#
# For full reproduction:
#   → use the complete dataset (Zenodo)
#   → run the full pipeline locally
#
# ============================================================

In [16]:
# Core pipeline/constructs the final event-level representation(FES)
#
# It integrates three independent components into a single aligned dataset:
# 1) Boundary events (J) extracted from co-reconstruction
# 2) Neural–quantum dynamics (E, Q) from Ricci time series
# 3) A reconstructed state variable (phi) derived from point-level h
#
# The procedure is:
# - Reconstruct a latent state variable phi on the points axis from h,
#   using a recursive formulation that combines:
#     • normalized boundary coordinate (h)
#     • smoothed boundary-event field (J_tilde)
#     • a gate function g_t depending on |h|
#
# - Map this point-level phi to the task axis and aggregate it per task
#
# - Align all quantities at the event level:
#     (J, phi, dphi, E, Q, dE, dQ, phase, distance, local correlation)
#
# - Standardize features within each session
#
# - Perform clustering in the resulting feature space to identify
#   the observed state structure in the data
#
# - Assign each cluster to one of the five theoretical states (FES),
#   based on predefined rules linking physical quantities to states
#
# The output of this cell is:
# - A complete event-level table describing system dynamics
# - A mapping between observed clusters and theoretical states (FES)
#
# This enables:
# - Direct analysis of how boundary events (J) relate to system dynamics (E, Q)
# - Identification of discrete state organization in the data
# - Validation of the theoretical five-state structure (FES) against observations
# ============================================================
import os
import re
import json
import glob
import numpy as np
import pandas as pd

from scipy.stats import pearsonr
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================================
# CONFIG
# =========================================
EVENT_CSV = "IDPC_Reproduction/J_dh_kappa_pooled_v2.csv"
RICCI_DIR = "IDPC_Reproduction_ricci"
POINTS_DIR = "IDPC_Reproduction"
OUTDIR = "IDPC_Reproduction"
os.makedirs(OUTDIR, exist_ok=True)

LOCAL_WIN = 2
MAX_LAG = 2
N_CLUSTERS = 5
RANDOM_STATE = 42

# ---- hybrid phi config ----
ALPHA = 0.7
TAU = 3.0
KAPPA = 1.0
HYBRID_EPS = 1e-8

# ---- boundary-event config (same logic origin as phi_next_event_table) ----
W_PRE  = 1
W_POST = 2
EPS_SCALE = 0.5
EPS_MODE  = "local"   # "local" or "global"
SMOOTH_RICCI = 2

# =========================================
# HELPERS
# =========================================
def normalize_label(x):
    s = str(x)
    m = re.search(r"P(1[0-9]|2[0-6]|[1-9])(?=_|$)", s)
    return f"P{m.group(1)}" if m else s

def pick_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def corr(x, y, min_n=4):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < min_n:
        return np.nan
    if np.std(x[m]) < 1e-12 or np.std(y[m]) < 1e-12:
        return np.nan
    return pearsonr(x[m], y[m])[0]

def best_local_corr(E, Q, center_idx, win=2, max_lag=2):
    s = max(0, center_idx - win)
    e = min(len(E), center_idx + win + 1)
    Ee = np.asarray(E[s:e], float)
    Qq = np.asarray(Q[s:e], float)

    best = np.nan
    best_lag = np.nan
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            x = Ee[:lag]
            y = Qq[-lag:]
        elif lag > 0:
            x = Ee[lag:]
            y = Qq[:-lag]
        else:
            x = Ee
            y = Qq

        r = corr(x, y, min_n=4)
        if np.isnan(r):
            continue
        if np.isnan(best) or abs(r) > abs(best):
            best = r
            best_lag = lag
    return best, best_lag

def zscore(x):
    x = np.asarray(x, float)
    m = np.isfinite(x)
    out = np.full_like(x, np.nan, dtype=float)
    if m.sum() < 3:
        return out
    mu = np.nanmean(x[m])
    sd = np.nanstd(x[m])
    if not np.isfinite(sd) or sd == 0:
        return out
    out[m] = (x[m] - mu) / sd
    return out

def circular_phase(dE, dQ):
    return np.arctan2(dQ, dE)

def list_labels(indir):
    labs = []
    for f in os.listdir(indir):
        if re.fullmatch(r"P([1-9]|1[0-9]|2[0-6])_timeseries\.csv", f):
            labs.append(f.replace("_timeseries.csv", ""))
    return sorted(labs, key=lambda s: int(s[1:]))

def moving_avg_same(x, W):
    x = np.asarray(x, float)
    if W <= 1:
        return x.copy()
    k = np.ones(int(W), float) / float(W)
    return np.convolve(x, k, mode="same")

def finite_diff(x):
    x = np.asarray(x, float)
    dx = np.full_like(x, np.nan, dtype=float)
    if len(x) < 2:
        return dx
    for i in range(1, len(x) - 1):
        if np.isfinite(x[i - 1]) and np.isfinite(x[i + 1]):
            dx[i] = 0.5 * (x[i + 1] - x[i - 1])
    if len(x) >= 2 and np.isfinite(x[0]) and np.isfinite(x[1]):
        dx[0] = x[1] - x[0]
    if len(x) >= 2 and np.isfinite(x[-1]) and np.isfinite(x[-2]):
        dx[-1] = x[-1] - x[-2]
    return dx

def gaussian_delta_eps(h, eps):
    h = np.asarray(h, float)
    eps = float(max(eps, 1e-12))
    return np.exp(-(h**2)/(2*eps**2)) / (np.sqrt(2*np.pi)*eps)

def zero_crossings(h):
    h = np.asarray(h, float)
    m = np.isfinite(h)
    if np.sum(m) < 2:
        return np.array([], dtype=int)
    hh = h.copy()
    if not m.all():
        idx = np.arange(len(hh))
        hh[~m] = np.interp(idx[~m], idx[m], hh[m])
    s = np.sign(hh)
    return np.where((s[1:] * s[:-1]) < 0)[0]

def build_hybrid_phi_from_points(
    h,
    alpha=0.7,
    tau=3.0,
    kappa=1.0,
    hybrid_eps=1e-8,
    w_pre=1,
    w_post=2,
    eps_scale=0.5,
    eps_mode="local",
):
    h = np.asarray(h, float)
    n = len(h)

    phi_full = np.full(n, np.nan, dtype=float)
    J_tilde = np.full(n, np.nan, dtype=float)
    g_full = np.full(n, np.nan, dtype=float)
    J_event = np.full(n, np.nan, dtype=float)

    if n < 5:
        return phi_full, J_tilde, g_full, J_event, np.array([], dtype=int)

    dh = finite_diff(h)
    flips = zero_crossings(h)

    # ---- build sparse J_event(k) at zero crossings ----
    h_std_global = np.nanstd(h)
    eps_global = eps_scale * h_std_global if np.isfinite(h_std_global) and h_std_global > 0 else 1e-6

    for k in flips:
        lo = max(0, k - w_pre)
        hi = min(n, k + w_post + 1)

        hw = h[lo:hi]
        dhw = dh[lo:hi]

        if np.sum(np.isfinite(hw)) < 3:
            continue

        if eps_mode == "local":
            loc_std = np.nanstd(hw)
            eps_loc = eps_scale * loc_std if np.isfinite(loc_std) and loc_std > 0 else eps_global
        else:
            eps_loc = eps_global

        delta_eps = gaussian_delta_eps(hw, eps_loc)
        phi_hat = dhw * delta_eps
        J_hat = np.nansum(phi_hat)

        J_event[k] = J_hat

    # ---- smooth sparse J_event -> J_tilde(t) over full points axis ----
    t = np.arange(n, dtype=float)
    event_idx = np.where(np.isfinite(J_event))[0]

    if len(event_idx) == 0:
        J_tilde[:] = 0.0
    else:
        num = np.zeros(n, dtype=float)
        den = np.zeros(n, dtype=float)

        for k in event_idx:
            w = np.exp(-((t - k) ** 2) / (2 * tau**2))
            num += J_event[k] * w
            den += w

        J_tilde = num / (den + hybrid_eps)

    # ---- scales ----
    s_h = np.nanstd(h)
    if not np.isfinite(s_h) or s_h < 1e-12:
        s_h = 1.0

    s_J = np.nanstd(J_tilde[np.isfinite(J_tilde)])
    if not np.isfinite(s_J) or s_J < 1e-12:
        s_J = 1.0

    rho = np.nanstd(np.abs(h))
    if not np.isfinite(rho) or rho < 1e-12:
        rho = 1.0

    # ---- gate ----
    g_full = np.exp(-np.abs(h) / rho)

    # ---- recursive hybrid phi ----
    base0 = (1.0 - g_full[0]) * (h[0] / s_h) + g_full[0] * kappa * (J_tilde[0] / s_J)
    phi_full[0] = base0

    for i in range(1, n):
        base = (1.0 - g_full[i]) * (h[i] / s_h) + g_full[i] * kappa * (J_tilde[i] / s_J)
        phi_full[i] = alpha * phi_full[i - 1] + (1.0 - alpha) * base

    return phi_full, J_tilde, g_full, J_event, flips

# =========================================
# LOAD SESSION-LEVEL TRUE RICCI SERIES
# =========================================
series_by_label = {}

for lab in list_labels(RICCI_DIR):
    fp = os.path.join(RICCI_DIR, f"{lab}_timeseries.csv")
    ts = pd.read_csv(fp)

    if not {"task_idx", "E_Ricci", "Q_Ricci"}.issubset(ts.columns):
        continue

    dat = ts[["task_idx", "E_Ricci", "Q_Ricci"]].copy()
    dat = dat.rename(columns={"E_Ricci": "E", "Q_Ricci": "Q"})
    dat = dat.sort_values("task_idx").reset_index(drop=True)

    dat["task_idx"] = pd.to_numeric(dat["task_idx"], errors="coerce")
    dat["E"] = pd.to_numeric(dat["E"], errors="coerce")
    dat["Q"] = pd.to_numeric(dat["Q"], errors="coerce")

    dat["dE"] = dat["E"].diff()
    dat["dQ"] = dat["Q"].diff()
    dat["phase"] = circular_phase(dat["dE"], dat["dQ"])
    dat["distance"] = np.sqrt((dat["E"] - dat["Q"])**2 + (dat["dE"] - dat["dQ"])**2)

    series_by_label[lab] = dat.copy()

print("loaded session series:", len(series_by_label))

# =========================================
# LOAD EVENT TABLE
# =========================================
ev = pd.read_csv(EVENT_CSV)

j_col = next((c for c in ["J", "J_obs", "boundary_impulse", "impulse", "J_hat"] if c in ev.columns), None)
task_col = next((c for c in ["event_task", "task_idx", "t_idx", "task_curr"] if c in ev.columns), None)

if j_col is None:
    raise KeyError(f"J column not found in {EVENT_CSV}: {list(ev.columns)}")
if task_col is None:
    raise KeyError(f"task column not found in {EVENT_CSV}: {list(ev.columns)}")
if "label" not in ev.columns:
    raise KeyError(f"label column not found in {EVENT_CSV}: {list(ev.columns)}")

ev = ev.rename(columns={j_col: "J", task_col: "task_idx"}).copy()
ev["label"] = ev["label"].map(normalize_label)
ev["task_idx"] = pd.to_numeric(ev["task_idx"], errors="coerce")

# =========================================
# BUILD phi TABLE FROM POINTS (HYBRID VERSION)
# =========================================
phi_task_rows = []

point_files = sorted(glob.glob(os.path.join(POINTS_DIR, "*_points.csv")))

for pts_path in point_files:
    label_raw = os.path.basename(pts_path).replace("_points.csv", "")
    label = normalize_label(label_raw)

    if label not in series_by_label:
        continue

    pts = pd.read_csv(pts_path)

    if "valid" in pts.columns:
        pts = pts[pd.to_numeric(pts["valid"], errors="coerce").fillna(0).to_numpy(int) == 1].copy()

    if "h" not in pts.columns:
        continue

    h = pd.to_numeric(pts["h"], errors="coerce").to_numpy(float)
    m = np.isfinite(h)
    pts = pts.loc[m].copy().reset_index(drop=True)
    h = h[m]

    if len(h) < 20:
        continue

    # ---- hybrid phi on points axis ----
    phi_full, J_tilde, g_full, J_event, flips = build_hybrid_phi_from_points(
        h=h,
        alpha=ALPHA,
        tau=TAU,
        kappa=KAPPA,
        hybrid_eps=HYBRID_EPS,
        w_pre=W_PRE,
        w_post=W_POST,
        eps_scale=EPS_SCALE,
        eps_mode=EPS_MODE,
    )

    pts["phi"] = phi_full
    pts["J_tilde"] = J_tilde
    pts["g_t"] = g_full
    pts["J_event"] = J_event

    # ---- same points -> task mapping logic as before ----
    T_task = len(series_by_label[label])
    n_points = len(pts)

    if T_task < 2 or n_points < 2:
        continue

    pts["point_order"] = np.arange(n_points)
    pts["task_idx"] = ((pts["point_order"] / n_points) * T_task).astype(int)
    pts["task_idx"] = pts["task_idx"].clip(lower=0, upper=T_task - 1)

    # ---- task-level average of phi ----
    agg = (
        pts.groupby("task_idx", as_index=False)[["phi", "J_tilde", "g_t"]]
        .mean()
        .sort_values("task_idx")
        .reset_index(drop=True)
    )

    agg["dphi"] = agg["phi"].diff()
    agg["label"] = label

    phi_task_rows.append(agg[["label", "task_idx", "phi", "dphi", "J_tilde", "g_t"]])

if len(phi_task_rows) == 0:
    raise RuntimeError("No phi task rows were built from *_points.csv")

phi_task = pd.concat(phi_task_rows, ignore_index=True)
phi_task["task_idx"] = pd.to_numeric(phi_task["task_idx"], errors="coerce")

print("loaded phi task rows =", len(phi_task))
print("phi labels =", phi_task["label"].nunique())

# lookup
phi_lookup = {}
for _, r in phi_task.iterrows():
    if pd.isna(r["task_idx"]):
        continue
    phi_lookup[(str(r["label"]), int(r["task_idx"]))] = {
        "phi": float(r["phi"]) if np.isfinite(r["phi"]) else np.nan,
        "dphi": float(r["dphi"]) if np.isfinite(r["dphi"]) else np.nan,
        "J_tilde": float(r["J_tilde"]) if np.isfinite(r["J_tilde"]) else np.nan,
        "g_t": float(r["g_t"]) if np.isfinite(r["g_t"]) else np.nan,
    }

# coverage check
match_count = 0
for _, r in ev.iterrows():
    if pd.isna(r["task_idx"]):
        continue
    key = (str(r["label"]), int(r["task_idx"]))
    if key in phi_lookup:
        match_count += 1

print("matched phi keys =", match_count, "/", len(ev))

# =========================================
# BUILD EVENT-LEVEL TABLE
# =========================================
rows = []

for _, r in ev.iterrows():
    lab = str(r["label"])
    if lab not in series_by_label:
        continue
    if pd.isna(r["task_idx"]):
        continue

    task_idx = int(r["task_idx"])
    dat = series_by_label[lab]

    hit = dat.loc[dat["task_idx"].astype(int) == task_idx]
    if len(hit) != 1:
        continue
    row = hit.iloc[0]

    center = dat.index[dat["task_idx"].astype(int) == task_idx][0]
    r_local, lag_local = best_local_corr(
        dat["E"].to_numpy(float),
        dat["Q"].to_numpy(float),
        center_idx=center,
        win=LOCAL_WIN,
        max_lag=MAX_LAG
    )

    phi_val = np.nan
    dphi_val = np.nan
    J_tilde_val = np.nan
    g_t_val = np.nan

    key = (lab, task_idx)
    if key in phi_lookup:
        phi_val = phi_lookup[key]["phi"]
        dphi_val = phi_lookup[key]["dphi"]
        J_tilde_val = phi_lookup[key]["J_tilde"]
        g_t_val = phi_lookup[key]["g_t"]

    rows.append({
        "label": lab,
        "task_idx": task_idx,
        "J": float(r["J"]),
        "phi": float(phi_val) if np.isfinite(phi_val) else np.nan,
        "dphi": float(dphi_val) if np.isfinite(dphi_val) else np.nan,
        "J_tilde": float(J_tilde_val) if np.isfinite(J_tilde_val) else np.nan,
        "g_t": float(g_t_val) if np.isfinite(g_t_val) else np.nan,
        "E": float(row["E"]) if np.isfinite(row["E"]) else np.nan,
        "Q": float(row["Q"]) if np.isfinite(row["Q"]) else np.nan,
        "dE": float(row["dE"]) if np.isfinite(row["dE"]) else np.nan,
        "dQ": float(row["dQ"]) if np.isfinite(row["dQ"]) else np.nan,
        "phase": float(row["phase"]) if np.isfinite(row["phase"]) else np.nan,
        "distance": float(row["distance"]) if np.isfinite(row["distance"]) else np.nan,
        "r_local": float(r_local) if np.isfinite(r_local) else np.nan,
        "lag_local": float(lag_local) if np.isfinite(lag_local) else np.nan,
    })

edf = pd.DataFrame(rows)

print("event rows =", len(edf))
print("labels =", edf["label"].nunique() if len(edf) else 0)

print("\n=== finite counts before z ===")
for c in ["phi", "dphi", "J", "phase", "distance", "r_local", "J_tilde", "g_t"]:
    n_ok = np.isfinite(pd.to_numeric(edf[c], errors="coerce")).sum() if c in edf.columns else 0
    print(f"{c:10s} = {n_ok} / {len(edf)}")

# session-wise z
for col in ["phi", "dphi", "J", "distance", "phase", "r_local", "J_tilde", "g_t"]:
    zcol = f"{col}_z"
    edf[zcol] = np.nan
    for lab, g in edf.groupby("label"):
        idx = g.index
        edf.loc[idx, zcol] = zscore(g[col].to_numpy(float))

raw_csv = os.path.join(OUTDIR, "event_level_raw_table_TRUE_RICCI__HYBRID_PHI.csv")
edf.to_csv(raw_csv, index=False)

print("\n=== finite counts after z ===")
for c in ["phi_z", "dphi_z", "J_z", "phase_z", "distance_z", "r_local_z", "J_tilde_z", "g_t_z"]:
    n_ok = np.isfinite(pd.to_numeric(edf[c], errors="coerce")).sum() if c in edf.columns else 0
    print(f"{c:10s} = {n_ok} / {len(edf)}")

# =========================================
# CLUSTERING FEATURES（既存構成維持）
# =========================================
feat_cols = [
    "phi_z",
    "dphi_z",
    "J_z",
    "phase_z",
    "distance_z",
    "r_local_z",
]

X = edf[feat_cols].copy()
mask = np.isfinite(X).all(axis=1)

edf_fit = edf.loc[mask].copy()
X_fit = X.loc[mask].to_numpy(float)

print("usable events for clustering =", len(edf_fit))

if len(edf_fit) == 0:
    raise RuntimeError(
        "No usable events for clustering. "
        "Hybrid phi mapping from points to task_idx did not produce matched finite rows."
    )

scaler = StandardScaler()
Xs = scaler.fit_transform(X_fit)

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=50
)
cluster_id = kmeans.fit_predict(Xs)
edf_fit["cluster"] = cluster_id

# PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Xp = pca.fit_transform(Xs)
edf_fit["pc1"] = Xp[:, 0]
edf_fit["pc2"] = Xp[:, 1]

# merge back
edf["cluster"] = np.nan
edf["pc1"] = np.nan
edf["pc2"] = np.nan
edf.loc[edf_fit.index, "cluster"] = edf_fit["cluster"]
edf.loc[edf_fit.index, "pc1"] = edf_fit["pc1"]
edf.loc[edf_fit.index, "pc2"] = edf_fit["pc2"]

# =========================================
# CLUSTER SUMMARY
# =========================================
cluster_summary = (
    edf_fit.groupby("cluster")[feat_cols + ["J", "phi", "dphi", "distance", "phase", "r_local", "J_tilde", "g_t"]]
    .mean()
    .reset_index()
)

cluster_counts = edf_fit.groupby("cluster").size().reset_index(name="n_events")
cluster_summary = cluster_summary.merge(cluster_counts, on="cluster", how="left")

cluster_csv = os.path.join(OUTDIR, "cluster_summary_TRUE_RICCI__HYBRID_PHI.csv")
cluster_summary.to_csv(cluster_csv, index=False)

clustered_csv = os.path.join(OUTDIR, "event_level_with_clusters_TRUE_RICCI__HYBRID_PHI.csv")
edf.to_csv(clustered_csv, index=False)

print("\n=== CLUSTER SUMMARY ===")
print(cluster_summary.sort_values("cluster").to_string(index=False))

# =========================================
# FES assignment (priority-based + fallback to next candidate + candidate ranking log)
# =========================================
tmp = cluster_summary.set_index("cluster")

# Evaluation feature for each FES state
# True  -> larger is better
# False -> smaller is better
fes_rules = {
    "CoCreation": ("r_local_z", True),
    "Surprise": ("J_z", True),        
    "SelfGrowth": ("dphi_z", True),
    "Challenge": ("distance_z", True),
    "Activation": ("phi_z", False),   
}

priority = ["CoCreation", "Surprise", "SelfGrowth", "Challenge", "Activation"]

# -------------------------
# Build candidate ranking lists
# -------------------------
candidate_lists = {}
candidate_scores = {}

for fes_name, (col, descending) in fes_rules.items():
    if fes_name == "Surprise":
        s = tmp["J_z"].abs().copy()
    elif fes_name == "Activation":
        s = tmp["phi_z"].abs().copy()
    else:
        s = tmp[col].copy()

    ordered = list(s.sort_values(ascending=not descending).index)
    candidate_lists[fes_name] = ordered
    candidate_scores[fes_name] = s.to_dict()

# -------------------------
# Assign clusters in priority order
# Select the highest-ranked unused candidate
# -------------------------
assigned = set()
final_map = {}
assignment_log_rows = []

for fes_name in priority:
    chosen_cluster = None
    chosen_rank = None
    chosen_score = np.nan

    for rank_idx, cl in enumerate(candidate_lists[fes_name], start=1):
        if cl not in assigned:
            chosen_cluster = cl
            chosen_rank = rank_idx
            chosen_score = candidate_scores[fes_name].get(cl, np.nan)
            break

    if chosen_cluster is not None:
        final_map[chosen_cluster] = fes_name
        assigned.add(chosen_cluster)

        assignment_log_rows.append({
            "fes_phase": fes_name,
            "assigned_cluster": int(chosen_cluster),
            "candidate_rank_used": int(chosen_rank),
            "score_used": float(chosen_score) if np.isfinite(chosen_score) else np.nan,
            "all_candidates_in_order": ",".join(map(str, candidate_lists[fes_name])),
            "selection_status": "selected",
        })
    else:
        assignment_log_rows.append({
            "fes_phase": fes_name,
            "assigned_cluster": np.nan,
            "candidate_rank_used": np.nan,
            "score_used": np.nan,
            "all_candidates_in_order": ",".join(map(str, candidate_lists[fes_name])),
            "selection_status": "no_available_cluster",
        })

# -------------------------
# Assign remaining clusters as "Unassigned"
# (Activation is not treated as a fallback category)
# -------------------------
for cl in tmp.index:
    if cl not in final_map:
        final_map[cl] = "Unassigned"

edf["fes_phase"] = edf["cluster"].map(final_map)

# -------------------------
# Output assignment log
# -------------------------
assignment_log = pd.DataFrame(assignment_log_rows)
assignment_log_csv = os.path.join(OUTDIR, "fes_assignment_log_TRUE_RICCI__HYBRID_PHI.csv")
assignment_log.to_csv(assignment_log_csv, index=False)

fes_csv = os.path.join(OUTDIR, "event_level_with_fes_phase_TRUE_RICCI.csv")
edf.to_csv(fes_csv, index=False)

fes_summary = (
    edf.groupby("fes_phase")[["J", "phi", "dphi", "distance", "phase", "r_local", "J_tilde", "g_t"]]
    .agg(["count", "mean", "std", "median"])
)
fes_summary_csv = os.path.join(OUTDIR, "fes_phase_summary_TRUE_RICCI__HYBRID_PHI.csv")
fes_summary.to_csv(fes_summary_csv)

print("\n=== CANDIDATE LISTS ===")
for fes_name in priority:
    print(f"{fes_name}: {candidate_lists[fes_name]}")

print("\n=== ASSIGNMENT LOG ===")
print(assignment_log.to_string(index=False))

print("\n=== FINAL FES MAPPING ===")
print(final_map)

print("\n=== FES PHASE SUMMARY ===")
print(fes_summary)

print("\nSaved:")
print(" -", os.path.basename(fes_csv))
print(" -", os.path.basename(fes_summary_csv))
print(" -", os.path.basename(assignment_log_csv))

# =========================================
# JSON SUMMARY
# =========================================
summary = {
    "n_events_total": int(len(edf)),
    "n_events_clustered": int(len(edf_fit)),
    "n_labels": int(edf["label"].nunique()),
    "feature_columns": feat_cols,
    "cluster_to_fes": {str(k): v for k, v in final_map.items()},
    "pca_explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "fes_assignment": "cluster_semantic_mapping",
    "phi_definition": "hybrid phi: phi_t = alpha*phi_{t-1} + (1-alpha)*[(1-g_t)*(h_t/s_h) + g_t*kappa*(J_tilde_t/s_J)]",
    "gate_definition": "g_t = exp(-abs(h_t)/rho)",
    "J_tilde_definition": "J_tilde_t = Gaussian smoothing of sparse boundary-event J_hat over points axis",
    "phi_task_mapping": "points axis mapped to task axis by task_idx = int((point_order / n_points) * T_task), then averaged within (label, task_idx)",
    "alpha": ALPHA,
    "tau": TAU,
    "kappa": KAPPA,
    "hybrid_eps": HYBRID_EPS,
    "w_pre": W_PRE,
    "w_post": W_POST,
    "eps_scale": EPS_SCALE,
    "eps_mode": EPS_MODE,
}

json_path = os.path.join(OUTDIR, "fes_event_embedding_summary_TRUE_RICCI__HYBRID_PHI.json")
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved to:", OUTDIR)
print(" -", os.path.basename(raw_csv))
print(" -", os.path.basename(clustered_csv))
print(" -", os.path.basename(fes_csv))
print(" -", os.path.basename(cluster_csv))
print(" -", os.path.basename(fes_summary_csv))
print(" -", os.path.basename(json_path))

loaded session series: 26
loaded phi task rows = 779
phi labels = 26
matched phi keys = 226 / 226
event rows = 226
labels = 26

=== finite counts before z ===
phi        = 226 / 226
dphi       = 226 / 226
J          = 226 / 226
phase      = 226 / 226
distance   = 226 / 226
r_local    = 226 / 226
J_tilde    = 226 / 226
g_t        = 226 / 226

=== finite counts after z ===
phi_z      = 226 / 226
dphi_z     = 226 / 226
J_z        = 226 / 226
phase_z    = 226 / 226
distance_z = 226 / 226
r_local_z  = 226 / 226
J_tilde_z  = 226 / 226
g_t_z      = 226 / 226
usable events for clustering = 226

=== CLUSTER SUMMARY ===
 cluster     phi_z    dphi_z       J_z   phase_z  distance_z  r_local_z         J       phi      dphi  distance     phase   r_local   J_tilde      g_t  n_events
       0 -0.642380 -0.746955  0.145431 -1.371586    1.042523  -0.243094  0.029197 -0.178255 -0.488605  9.178610 -3.023824 -0.193110 -0.103023 0.441727        30
       1 -0.366631 -0.343274  0.078175  1.278686    1.010833

In [ ]:
# ============================================================
# Chapter 2 Figure Generation Script
# ------------------------------------------------------------
# Purpose:
# This script generates all figures presented in Chapter 2 of
# the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:0‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# Chapter 2 demonstrates that structural agreement emerges
# between two physically independent systems:
#   - Neural dynamics (EEG-derived Ricci curvature: E)
#   - Quantum dynamics (probability-geometry-derived Ricci curvature: Q)
#
# Importantly, this agreement is NOT evaluated via direct correlation
# of raw time series. Instead:
#   - Each system is independently reconstructed using a one-step
#     predictive framework
#   - Agreement is evaluated between reconstructed dynamics
#
# The goal is to show that:
#   - Structural agreement exists without causal interaction
#   - The agreement is selective (not uniform over time)
#   - Multiple structural constraints co-occur:
#       • temporal correspondence
#       • discrete state structure
#       • non-random distribution
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from scipy.stats import pearsonr

# =========================
# Configuration
# =========================
INDIR = "IDPC_Reproduction"
OUTDIR = os.path.join(INDIR, "Chapter2")
os.makedirs(OUTDIR, exist_ok=True)

INPUT_CSV = os.path.join(INDIR, "event_level_with_fes_phase_TRUE_RICCI.csv")

CONF_MARGIN = 0.00
USE_CONF_FILTER = False
BEST_MIN_N = 5

RIDGE_ALPHA_Q = 1.0

# =========================
# Helper functions
# =========================
def safe_pearson(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 3:
        return np.nan
    if np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12:
        return np.nan
    return pearsonr(x, y)[0]

def build_train_for_source(df_source):
    rows = []

    for lab, g in df_source.groupby("label"):
        g = g.sort_values("task_idx").reset_index(drop=True)

        for i in range(1, len(g) - 1):
            s_next = g.loc[i + 1, "fes_phase"]
            if s_next not in ["Surprise", "CoCreation"]:
                continue

            vals = [
                g.loc[i, "phi"], g.loc[i, "dphi"],
                g.loc[i, "E"], g.loc[i, "Q"],
                g.loc[i - 1, "E"], g.loc[i - 1, "Q"],
                g.loc[i + 1, "E"], g.loc[i + 1, "Q"]
            ]
            if not np.all(np.isfinite(vals)):
                continue

            E_t = g.loc[i, "E"]
            Q_t = g.loc[i, "Q"]
            E_tm1 = g.loc[i - 1, "E"]
            Q_tm1 = g.loc[i - 1, "Q"]

            dE_t = E_t - E_tm1
            dQ_t = Q_t - Q_tm1

            E_next = g.loc[i + 1, "E"]
            Q_next = g.loc[i + 1, "Q"]

            dQ_next = Q_next - Q_t

            if dQ_next > 0:
                y_sign_q = 1
            elif dQ_next < 0:
                y_sign_q = -1
            else:
                y_sign_q = 0

            rows.append({
                "label": lab,
                "task_idx_t": g.loc[i, "task_idx"],
                "task_idx_next": g.loc[i + 1, "task_idx"],
                "phi_t": g.loc[i, "phi"],
                "dphi_t": g.loc[i, "dphi"],
                "E_t": E_t,
                "Q_t": Q_t,
                "E_tm1": E_tm1,
                "Q_tm1": Q_tm1,
                "dE_t": dE_t,
                "dQ_t": dQ_t,
                "state_next": s_next,
                "y_state": 1 if s_next == "Surprise" else 0,
                "E_next": E_next,
                "Q_next": Q_next,
                "dQ_next": dQ_next,
                "y_sign_q": y_sign_q,
            })

    return pd.DataFrame(rows).dropna().reset_index(drop=True)

# =========================
# Data loading
# =========================
df = pd.read_csv(INPUT_CSV)
df = df.sort_values(["label", "task_idx"]).reset_index(drop=True)

needed = ["label", "task_idx", "phi", "dphi", "E", "Q", "fes_phase"]
for c in needed:
    if c not in df.columns:
        raise KeyError(f"missing column: {c}")

for c in ["phi", "dphi", "E", "Q"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

all_labels = sorted(df["label"].dropna().unique())

# =========================
# LOSO state prediction & per-session evaluation
# =========================
session_rows = []
session_demos = {}
state_eval_rows = []
state_perf_rows = []
dyn_coef_rows = []

for lab in all_labels:
    train_source = df[df["label"] != lab].copy()
    test_g = df[df["label"] == lab].sort_values("task_idx").reset_index(drop=True)

    train = build_train_for_source(train_source)
    if len(train) < 10:
        continue

    # -------------------------
    # State prediction model
    # -------------------------
    X_state_df = train[["phi_t", "dphi_t"]].copy()
    y_state = train["y_state"].values

    state_scaler = StandardScaler()
    X_state_scaled = state_scaler.fit_transform(X_state_df)

    state_clf = LogisticRegression(max_iter=1000)
    state_clf.fit(X_state_scaled, y_state)

    p_train = state_clf.predict_proba(X_state_scaled)[:, 1]
    y_pred_train = (p_train >= 0.5).astype(int)

    acc = float((y_pred_train == y_state).mean())
    auc = float(roc_auc_score(y_state, p_train))

    state_perf_rows.append({
        "test_label": lab,
        "accuracy_train": acc,
        "auc_train": auc,
        "n_train": len(train)
    })

    # -------------------------
    # AR(2)+Δ dynamics per state
    # + Q sign model
    # -------------------------
    dyn_models = {}
    dyn_rows_this = []

    for st in ["CoCreation", "Surprise"]:
        sub = train[train["state_next"] == st].copy()
        if len(sub) < 3:
            continue

        XE = sub[["E_t", "E_tm1", "dE_t"]].values
        yE = sub["E_next"].values
        model_E = LinearRegression()
        model_E.fit(XE, yE)

        XQ = sub[["Q_t", "Q_tm1", "dQ_t"]].values
        yQ = sub["Q_next"].values
        model_Q = Ridge(alpha=RIDGE_ALPHA_Q)
        model_Q.fit(XQ, yQ)

        sub_sign = sub[sub["y_sign_q"] != 0].copy()
        sign_model_Q = None
        sign_acc_train = np.nan

        if len(sub_sign) >= 4 and len(np.unique(sub_sign["y_sign_q"])) >= 2:
            XQ_sign = sub_sign[["Q_t", "Q_tm1", "dQ_t"]].values
            yQ_sign = (sub_sign["y_sign_q"] > 0).astype(int).values

            sign_model_Q = LogisticRegression(max_iter=1000)
            sign_model_Q.fit(XQ_sign, yQ_sign)

            sign_pred_train = sign_model_Q.predict(XQ_sign)
            sign_acc_train = float(np.mean(sign_pred_train == yQ_sign))

        dyn_models[st] = {
            "E_model": model_E,
            "Q_model": model_Q,
            "Q_sign_model": sign_model_Q,
            "n": len(sub),
            "E_coef": (model_E.coef_.copy(), model_E.intercept_),
            "Q_coef": (model_Q.coef_.copy(), model_Q.intercept_),
            "Q_sign_acc_train": sign_acc_train,
        }

        aE, bE = dyn_models[st]["E_coef"]
        aQ, bQ = dyn_models[st]["Q_coef"]

        dyn_rows_this.append({
            "test_label": lab,
            "state": st,
            "n": len(sub),
            "E_a1": aE[0],
            "E_a2": aE[1],
            "E_a3": aE[2],
            "E_b": bE,
            "Q_c1": aQ[0],
            "Q_c2": aQ[1],
            "Q_c3": aQ[2],
            "Q_d": bQ,
            "Q_sign_acc_train": sign_acc_train,
        })

    if len(dyn_rows_this):
        dyn_coef_rows.append(pd.DataFrame(dyn_rows_this))

    # -------------------------
    # State prediction & reconstruction on test session
    # -------------------------
    pred_rows = []

    for i in range(1, len(test_g) - 1):
        vals = [
            test_g.loc[i, "phi"], test_g.loc[i, "dphi"],
            test_g.loc[i, "E"], test_g.loc[i, "Q"],
            test_g.loc[i - 1, "E"], test_g.loc[i - 1, "Q"],
            test_g.loc[i + 1, "E"], test_g.loc[i + 1, "Q"]
        ]
        if not np.all(np.isfinite(vals)):
            continue

        x_state_df = pd.DataFrame([{
            "phi_t": test_g.loc[i, "phi"],
            "dphi_t": test_g.loc[i, "dphi"]
        }])
        x_state_scaled = state_scaler.transform(x_state_df)

        p_sur = state_clf.predict_proba(x_state_scaled)[0, 1]

        s_next = test_g.loc[i + 1, "fes_phase"]
        if s_next in ["Surprise", "CoCreation"]:
            state_eval_rows.append({
                "label": lab,
                "task_idx_next": test_g.loc[i + 1, "task_idx"],
                "y_true": 1 if s_next == "Surprise" else 0,
                "y_true_label": s_next,
                "p_surprise": float(p_sur),
                "y_pred": 1 if p_sur >= 0.5 else 0,
                "y_pred_label": "Surprise" if p_sur >= 0.5 else "CoCreation"
            })

        if USE_CONF_FILTER and abs(p_sur - 0.5) < CONF_MARGIN:
            continue

        state_pred = "Surprise" if p_sur >= 0.5 else "CoCreation"
        if state_pred not in dyn_models:
            continue

        E_t = test_g.loc[i, "E"]
        Q_t = test_g.loc[i, "Q"]
        E_tm1 = test_g.loc[i - 1, "E"]
        Q_tm1 = test_g.loc[i - 1, "Q"]

        dE_t = E_t - E_tm1
        dQ_t = Q_t - Q_tm1

        XE = np.array([[E_t, E_tm1, dE_t]])
        XQ = np.array([[Q_t, Q_tm1, dQ_t]])

        E_hat = dyn_models[state_pred]["E_model"].predict(XE)[0]

        Q_raw = dyn_models[state_pred]["Q_model"].predict(XQ)[0]
        dQ_raw = Q_raw - Q_t

        sign_model_Q = dyn_models[state_pred]["Q_sign_model"]
        if sign_model_Q is not None:
            sign_prob = sign_model_Q.predict_proba(XQ)[0, 1]
            sign_pred = 1 if sign_prob >= 0.5 else -1
            dQ_fixed = abs(dQ_raw) * sign_pred
            Q_hat = Q_t + dQ_fixed
        else:
            sign_prob = np.nan
            Q_hat = Q_raw

        pred_rows.append({
            "task": test_g.loc[i + 1, "task_idx"],
            "phi_t": test_g.loc[i, "phi"],
            "dphi_t": test_g.loc[i, "dphi"],
            "E_t": E_t,
            "Q_t": Q_t,
            "E_tm1": E_tm1,
            "Q_tm1": Q_tm1,
            "dE_t": dE_t,
            "dQ_t": dQ_t,
            "E_true": test_g.loc[i + 1, "E"],
            "Q_true": test_g.loc[i + 1, "Q"],
            "E_pred": E_hat,
            "Q_pred": Q_hat,
            "Q_raw_pred": Q_raw,
            "Q_sign_prob_positive": sign_prob,
            "p_surprise": p_sur,
            "state_pred": state_pred,
        })

    demo = pd.DataFrame(pred_rows)
    if len(demo) < 3:
        continue

    rE = safe_pearson(demo["E_true"], demo["E_pred"])
    rQ = safe_pearson(demo["Q_true"], demo["Q_pred"])
    rEQ = safe_pearson(demo["E_true"], demo["Q_true"])

    session_rows.append({
        "label": lab,
        "n": len(demo),
        "pearson_E": rE,
        "pearson_Q": rQ,
        "pearson_EQ": rEQ,
        "mean_abs_pearson": np.nanmean([abs(rE), abs(rQ)]),
    })

    session_demos[lab] = demo.copy()

pearson_table = pd.DataFrame(session_rows).sort_values("mean_abs_pearson", ascending=False)
if len(pearson_table) == 0:
    raise ValueError("No sessions survived evaluation.")

# Auxiliary column for representative selection
pearson_table["rep_score"] = (
    pearson_table["pearson_E"].clip(lower=0) +
    pearson_table["pearson_Q"].clip(lower=0)
) / 2.0

all_demo_df = pd.concat(
    [d.assign(label=lab) for lab, d in session_demos.items()],
    ignore_index=True
)

# =========================
# Figure 7: State model summary / confusion matrix
# =========================
state_eval_df = pd.DataFrame(state_eval_rows)

if len(state_eval_df):
    overall_acc = float((state_eval_df["y_true"] == state_eval_df["y_pred"]).mean())
    overall_auc = float(roc_auc_score(state_eval_df["y_true"], state_eval_df["p_surprise"]))
    state_model_perf_df = pd.DataFrame([{
        "overall_accuracy_test": overall_acc,
        "overall_auc_test": overall_auc,
        "n_state_test_rows": len(state_eval_df)
    }])
else:
    state_model_perf_df = pd.DataFrame([{
        "overall_accuracy_test": np.nan,
        "overall_auc_test": np.nan,
        "n_state_test_rows": 0
    }])

state_model_perf_df.to_csv(os.path.join(OUTDIR, "state_model_performance.csv"), index=False)

if len(state_eval_df):
    cm = confusion_matrix(state_eval_df["y_true"], state_eval_df["y_pred"])
    plt.figure(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(cm, display_labels=["CoCreation", "Surprise"])
    disp.plot(colorbar=False)
    plt.title("State prediction confusion matrix(LOSO)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, "state_confusion_matrix.png"), dpi=300)
    plt.show()
    plt.close()

print("\n=== STATE MODEL (LOSO) ===")
print(state_model_perf_df.to_string(index=False))

# =========================
# Save dynamics coefficients
# =========================
if len(dyn_coef_rows):
    dyn_coef_df = pd.concat(dyn_coef_rows, ignore_index=True)
    dyn_coef_df.to_csv(os.path.join(OUTDIR, "state_dynamics_coefficients.csv"), index=False)

    print("\n=== STATE-SPECIFIC AR(2)+Δ DYNAMICS (sample rows) ===")
    print(dyn_coef_df.head().to_string(index=False))

# =========================
# Figure 1: Overall bar plot
# =========================
plt.figure(figsize=(12, 4))
plt.bar(pearson_table["label"], pearson_table["mean_abs_pearson"])
plt.axhline(pearson_table["mean_abs_pearson"].mean(), linestyle="--")
plt.xticks(rotation=90)
plt.title("mean |Pearson| by session")
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "mean_abs_pearson_bar.png"), dpi=300)
plt.show()
plt.close()

import os
import math
import matplotlib.pyplot as plt

# =========================
# Figure 2: E panels
# =========================
import os
import math
import matplotlib.pyplot as plt

pearson_table_E = pearson_table.sort_values("pearson_E", ascending=False).reset_index(drop=True)
labels_plot_E = list(pearson_table_E["label"])

n_plots_E = len(labels_plot_E)
n_cols = 6
n_rows_E = math.ceil(n_plots_E / n_cols)

fig, axes = plt.subplots(n_rows_E, n_cols, figsize=(18, 3.0 * n_rows_E))
axes = axes.flatten()

for i, lab in enumerate(labels_plot_E):
    ax = axes[i]
    demo = session_demos[lab]

    ax.plot(demo["task"], demo["E_true"], linewidth=1.3)
    ax.plot(demo["task"], demo["E_pred"], linewidth=1.1)

    rE = pearson_table_E.loc[pearson_table_E["label"] == lab, "pearson_E"].iloc[0]
    n_ = pearson_table_E.loc[pearson_table_E["label"] == lab, "n"].iloc[0]

    # Restore original: include r and n in title
    ax.set_title(f"{lab} rE={rE:.3f}, n={n_}", fontsize=8)
    ax.grid(alpha=0.3)

for j in range(n_plots_E, len(axes)):
    fig.delaxes(axes[j])

fig.legend(["E_true", "E_pred"], loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(OUTDIR, "all_sessions_E_panels_sorted_by_rE.png"),
            dpi=300, bbox_inches="tight")
plt.show()
plt.close()


# =========================
# Figure 3: Q panels
# =========================
pearson_table_Q = pearson_table.sort_values("pearson_Q", ascending=False).reset_index(drop=True)
labels_plot_Q = list(pearson_table_Q["label"])

n_plots_Q = len(labels_plot_Q)
n_cols = 6
n_rows_Q = math.ceil(n_plots_Q / n_cols)

fig, axes = plt.subplots(n_rows_Q, n_cols, figsize=(18, 3.0 * n_rows_Q))
axes = axes.flatten()

for i, lab in enumerate(labels_plot_Q):
    ax = axes[i]
    demo = session_demos[lab]

    ax.plot(demo["task"], demo["Q_true"], linewidth=1.3)
    ax.plot(demo["task"], demo["Q_pred"], linewidth=1.1)

    rQ = pearson_table_Q.loc[pearson_table_Q["label"] == lab, "pearson_Q"].iloc[0]
    n_ = pearson_table_Q.loc[pearson_table_Q["label"] == lab, "n"].iloc[0]

    # Restore original: include r and n in title
    ax.set_title(f"{lab} rQ={rQ:.3f}, n={n_}", fontsize=8)
    ax.grid(alpha=0.3)

for j in range(n_plots_Q, len(axes)):
    fig.delaxes(axes[j])

fig.legend(["Q_true", "Q_pred"], loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(OUTDIR, "all_sessions_Q_panels_sorted_by_rQ.png"),
            dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# =========================
# Representative session selection (separate E and Q)
# =========================
best_E_candidates = pearson_table[
    (pearson_table["pearson_E"] > 0) &
    (pearson_table["n"] >= BEST_MIN_N)
].copy()

if len(best_E_candidates) == 0:
    best_E_candidates = pearson_table.copy()

best_E_lab = best_E_candidates.sort_values(
    "pearson_E", ascending=False
).iloc[0]["label"]

best_Q_candidates = pearson_table[
    (pearson_table["pearson_Q"] > 0) &
    (pearson_table["n"] >= BEST_MIN_N)
].copy()

if len(best_Q_candidates) == 0:
    best_Q_candidates = pearson_table.copy()

best_Q_lab = best_Q_candidates.sort_values(
    "pearson_Q", ascending=False
).iloc[0]["label"]

print("\n=== REPRESENTATIVE SESSIONS ===")
print("E best:", best_E_lab)
print("Q best:", best_Q_lab)

# =========================
# Figure 4: Combine auto-selected E-best and Q-best into one PNG
# =========================
demo_E = session_demos[best_E_lab]
demo_Q = session_demos[best_Q_lab]

rE_best = pearson_table.loc[
    pearson_table["label"] == best_E_lab, "pearson_E"
].iloc[0]

rQ_best = pearson_table.loc[
    pearson_table["label"] == best_Q_lab, "pearson_Q"
].iloc[0]

combined_png = os.path.join(OUTDIR, "Figure4_best_E_and_Q_combined.png")

plt.figure(figsize=(9, 7))

# -------------------------
# Top: Auto-selected E-best
# -------------------------
plt.subplot(2, 1, 1)
plt.plot(demo_E["task"], demo_E["E_true"], label="E_true", linewidth=2)
plt.plot(demo_E["task"], demo_E["E_pred"], "--", label="E_pred", linewidth=2)
plt.title(f"Neural Ricci ({best_E_lab}) | Pearson r = {rE_best:.3f}")
plt.ylabel("E_Ricci")
plt.legend()
plt.grid(alpha=0.3)

# -------------------------
# Bottom: Auto-selected Q-best
# -------------------------
plt.subplot(2, 1, 2)
plt.plot(demo_Q["task"], demo_Q["Q_true"], label="Q_true", linewidth=2)
plt.plot(demo_Q["task"], demo_Q["Q_pred"], "--", label="Q_pred", linewidth=2)
plt.title(f"Quantum Ricci ({best_Q_lab}) | Pearson r = {rQ_best:.3f}")
plt.xlabel("task")
plt.ylabel("Q_Ricci")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(combined_png, dpi=300)
plt.show()
plt.close()

# Save reference CSV if needed
demo_E.to_csv(os.path.join(OUTDIR, f"{best_E_lab}_E_best.csv"), index=False)
demo_Q.to_csv(os.path.join(OUTDIR, f"{best_Q_lab}_Q_best.csv"), index=False)

print("Saved:", combined_png)
print("E best:", best_E_lab)
print("Q best:", best_Q_lab)


# =========================
# Figure 5: Alignment scatter
# Select session with maximum r_aligned
# =========================
alignment_rows = []

for lab, demo in session_demos.items():
    E = demo["E_true"].values
    Q = demo["Q_true"].values

    m = np.isfinite(E) & np.isfinite(Q)
    E = E[m]
    Q = Q[m]

    if len(E) < BEST_MIN_N:
        continue

    align_model = LinearRegression()
    align_model.fit(Q.reshape(-1, 1), E)
    Q_aligned = align_model.predict(Q.reshape(-1, 1))
    r_aligned = safe_pearson(E, Q_aligned)

    row_src = pearson_table.loc[pearson_table["label"] == lab]
    rep_score = row_src["rep_score"].iloc[0] if len(row_src) else np.nan
    mean_abs_pearson = row_src["mean_abs_pearson"].iloc[0] if len(row_src) else np.nan

    alignment_rows.append({
        "label": lab,
        "n": len(E),
        "a": align_model.coef_[0],
        "b": align_model.intercept_,
        "r_aligned": r_aligned,
        "rep_score": rep_score,
        "mean_abs_pearson": mean_abs_pearson,
    })

alignment_table = pd.DataFrame(alignment_rows)

if len(alignment_table):
    best_row = alignment_table.sort_values(
        ["r_aligned", "rep_score", "mean_abs_pearson", "n"],
        ascending=[False, False, False, False]
    ).iloc[0]

    align_lab = best_row["label"]
    align_demo = session_demos[align_lab]

    E = align_demo["E_true"].values
    Q = align_demo["Q_true"].values
    m = np.isfinite(E) & np.isfinite(Q)
    E = E[m]
    Q = Q[m]

    align_model = LinearRegression()
    align_model.fit(Q.reshape(-1, 1), E)
    Q_aligned = align_model.predict(Q.reshape(-1, 1))
    r_aligned = safe_pearson(E, Q_aligned)

    alignment_table.to_csv(os.path.join(OUTDIR, "alignment_table_all_sessions.csv"), index=False)

    pd.DataFrame([best_row]).to_csv(
        os.path.join(OUTDIR, f"{align_lab}_alignment_summary.csv"),
        index=False
    )

    plt.figure(figsize=(6, 5))
    plt.scatter(E, Q_aligned)
    plt.xlabel("E_true")
    plt.ylabel("Q_aligned")
    plt.title(f"Best alignment ({align_lab}) | r = {r_aligned:.3f}")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f"{align_lab}_alignment_scatter.png"), dpi=300)
    plt.show()
    plt.close()

    print("\n=== BEST ALIGNMENT SESSION BY r_aligned ===")
    print(pd.DataFrame([best_row]).to_string(index=False))
else:
    print("\nNo alignment candidate found.")

# =========================
# Figure 6: Distribution plots
# =========================
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(pearson_table["pearson_E"].dropna(), bins=15)
plt.axvline(pearson_table["pearson_E"].mean(), linestyle="--")
plt.title("Distribution of Pearson E")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(pearson_table["pearson_Q"].dropna(), bins=15)
plt.axvline(pearson_table["pearson_Q"].mean(), linestyle="--")
plt.title("Distribution of Pearson Q")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "pearson_distributions.png"), dpi=300)
plt.show()
plt.close()

# =========================
# TOP / BOTTOM / SUMMARY
# =========================
print("\n=== SESSION PEARSON TABLE ===")
print(pearson_table.to_string(index=False))

print("\n=== SUMMARY ===")
print("mean Pearson E =", pearson_table["pearson_E"].mean())
print("mean Pearson Q =", pearson_table["pearson_Q"].mean())
print("mean Pearson EQ =", pearson_table["pearson_EQ"].mean())
print("mean abs Pearson =", pearson_table["mean_abs_pearson"].mean())

pearson_table.to_csv(os.path.join(OUTDIR, "pearson_table_dynamic.csv"), index=False)

top5 = pearson_table.head(5)
bot5 = pearson_table.tail(5)

print("\n=== TOP 5 ===")
print(top5.to_string(index=False))

print("\n=== BOTTOM 5 ===")
print(bot5.to_string(index=False))

top5.to_csv(os.path.join(OUTDIR, "top5_sessions.csv"), index=False)
bot5.to_csv(os.path.join(OUTDIR, "bottom5_sessions.csv"), index=False)

summary_df = pd.DataFrame([{
    "n_sessions_shown": len(pearson_table),
    "mean_Pearson_E": pearson_table["pearson_E"].mean(),
    "mean_Pearson_Q": pearson_table["pearson_Q"].mean(),
    "mean_Pearson_EQ": pearson_table["pearson_EQ"].mean(),
    "mean_abs_Pearson": pearson_table["mean_abs_pearson"].mean(),
    "state_test_accuracy": state_model_perf_df["overall_accuracy_test"].iloc[0],
    "state_test_auc": state_model_perf_df["overall_auc_test"].iloc[0],
}])
summary_df.to_csv(os.path.join(OUTDIR, "summary_metrics.csv"), index=False)

# =========================
# Save reconstruction CSV for all sessions
# =========================
all_demo_df.to_csv(os.path.join(OUTDIR, "all_sessions_dynamic_reconstruction.csv"), index=False)

dropped = [lab for lab in all_labels if lab not in pearson_table["label"].tolist()]
print("\nDropped labels:", dropped)

print("\nSaved outputs to:", OUTDIR)

In [ ]:
# ============================================================
# Figure 8 Generation Script for Chapter 3
# ------------------------------------------------------------
# Purpose:
# This script generates Figure 8 for Chapter 3 of the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:1‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# Figure 8 visualizes the discrete state structure underlying
# neural–quantum dynamics across sessions.
#
# In the paper, this figure is used to show that:
#   - session dynamics are organized in a common five-state space
#   - the state structure is discrete rather than continuous
#   - boundary impulse events appear at specific localized points
#   - structural alignment can appear independently of impulse events
#
# Each panel corresponds to one session and displays:
#   - state trajectory over task/event order
#   - FES state labels mapped to a common vertical state axis
#   - blue bar markers for J-based boundary impulse events
#   - orange outlines for independently detected alignment points
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

PROVEPHI_DIR = "IDPC_Reproduction"
RICCI_DIR = "IDPC_Reproduction_ricci"
EVENT_CSV = "IDPC_Reproduction/J_dh_kappa_pooled_v2.csv"
PHASE_CSV = "IDPC_Reproduction/event_level_with_fes_phase_TRUE_RICCI.csv"
OUTDIR = "IDPC_Reproduction/Chapter3"
os.makedirs(OUTDIR, exist_ok=True)

# FES order
fes_order = ["Activation", "Challenge", "Surprise", "SelfGrowth", "CoCreation"]
fes_to_y = {k: i for i, k in enumerate(fes_order)}

# Colors
colors = {
    "Activation": "#FFD700",
    "Challenge": "#32CD32",
    "Surprise": "#1E90FF",
    "SelfGrowth": "#FF0000",
    "CoCreation": "#FFA500",
}

# Alignment thresholds
PHASE_THRESH_DEG = 12.0   # |Δpsi| must be below this threshold
AMP_THRESH_Z = 0.75       # |E_z - Q_z| must be below this threshold

def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def zscore_series(x):
    x = pd.to_numeric(x, errors="coerce").to_numpy(float)
    out = np.full(len(x), np.nan, dtype=float)
    m = np.isfinite(x)
    if m.sum() < 2:
        return out
    mu = np.nanmean(x[m])
    sd = np.nanstd(x[m])
    if sd < 1e-12:
        out[m] = 0.0
    else:
        out[m] = (x[m] - mu) / sd
    return out

# =========================
# Load J events
# =========================
ev = pd.read_csv(EVENT_CSV)
j_col = next(c for c in ["J", "J_obs", "boundary_impulse", "impulse", "J_hat"] if c in ev.columns)
task_col = next(c for c in ["event_task", "task_idx", "t_idx", "task_curr"] if c in ev.columns)
ev = ev.rename(columns={j_col: "J", task_col: "task_idx"})
ev["task_idx"] = pd.to_numeric(ev["task_idx"], errors="coerce")

# =========================
# Load fixed FES labels
# =========================
phase_df = pd.read_csv(PHASE_CSV)
need = {"label", "task_idx", "fes_phase"}
missing = need - set(phase_df.columns)
if missing:
    raise KeyError(f"Missing columns in {PHASE_CSV}: {missing}")

phase_df = phase_df[["label", "task_idx", "fes_phase"]].copy()
phase_df["task_idx"] = pd.to_numeric(phase_df["task_idx"], errors="coerce")

# =========================
# Build session list
# =========================
labels = sorted([
    f.replace("_quantum_timeseries.csv", "")
    for f in os.listdir(PROVEPHI_DIR)
    if f.endswith("_quantum_timeseries.csv")
], key=lambda x: int(x[1:]))

# =========================
# Assign nearest FES label from event-level data
# =========================
def assign_nearest_fes(q_task_idx, event_task_idx, event_fes):
    q_task_idx = np.asarray(q_task_idx, dtype=float)
    event_task_idx = np.asarray(event_task_idx, dtype=float)
    event_fes = np.asarray(event_fes, dtype=object)

    assigned = []
    nearest_event_idx = []

    for t in q_task_idx:
        j = np.argmin(np.abs(event_task_idx - t))
        assigned.append(event_fes[j])
        nearest_event_idx.append(j)

    return np.array(assigned, dtype=object), np.array(nearest_event_idx, dtype=int)

# =========================
# Create figure
# =========================
n_cols = 6
n_rows = 5

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 12))
axes = axes.flatten()

for i, lab in enumerate(labels):
    ax = axes[i]

    q = pd.read_csv(os.path.join(PROVEPHI_DIR, f"{lab}_quantum_timeseries.csv"))
    if "task_idx" not in q.columns:
        raise KeyError(f"{lab}_quantum_timeseries.csv does not contain task_idx")

    q = q.sort_values("task_idx").reset_index(drop=True)
    q["task_idx"] = pd.to_numeric(q["task_idx"], errors="coerce")
    q["event_order"] = np.arange(1, len(q) + 1)

    # Assign FES labels from nearest event-level annotation
    sub_phase = phase_df[phase_df["label"] == lab].sort_values("task_idx").reset_index(drop=True)
    if len(sub_phase) == 0:
        q["fes_phase"] = None
        q["nearest_event_idx"] = np.nan
    else:
        assigned_fes, nearest_idx = assign_nearest_fes(
            q["task_idx"].values,
            sub_phase["task_idx"].values,
            sub_phase["fes_phase"].values
        )
        q["fes_phase"] = assigned_fes
        q["nearest_event_idx"] = nearest_idx

    xs = q["event_order"].values
    ys = np.array([fes_to_y.get(p, np.nan) for p in q["fes_phase"]])

    # =========================
    # Detect alignment points independently of J
    # =========================
    ricci_fp = os.path.join(RICCI_DIR, f"{lab}_timeseries.csv")
    highlight_task_set = set()

    if os.path.exists(ricci_fp):
        r = pd.read_csv(ricci_fp).sort_values("task_idx").reset_index(drop=True)

        e_col = "E_Ricci" if "E_Ricci" in r.columns else None
        q_col = "Q_Ricci_affine" if "Q_Ricci_affine" in r.columns else ("Q_Ricci" if "Q_Ricci" in r.columns else None)

        if e_col is not None and q_col is not None:
            r["task_idx"] = pd.to_numeric(r["task_idx"], errors="coerce")
            r["E"] = pd.to_numeric(r[e_col], errors="coerce")
            r["Q"] = pd.to_numeric(r[q_col], errors="coerce")

            r["E_z"] = zscore_series(r["E"])
            r["Q_z"] = zscore_series(r["Q"])

            dE = np.gradient(r["E"].to_numpy(float))
            dQ = np.gradient(r["Q"].to_numpy(float))

            psiE = np.arctan2(dE, r["E"].to_numpy(float))
            psiQ = np.arctan2(dQ, r["Q"].to_numpy(float))
            dpsi = wrap_pi(psiE - psiQ)

            r["dpsi_deg"] = np.degrees(np.abs(dpsi))
            r["amp_gap"] = np.abs(r["E_z"] - r["Q_z"])

            # Alignment condition
            align_mask = (
                np.isfinite(r["dpsi_deg"]) &
                np.isfinite(r["amp_gap"]) &
                (r["dpsi_deg"] <= PHASE_THRESH_DEG) &
                (r["amp_gap"] <= AMP_THRESH_Z)
            )

            highlight_task_set = set(r.loc[align_mask, "task_idx"].tolist())

    # ===== Trajectory lines =====
    for j in range(len(xs) - 1):
        if np.isnan(ys[j]) or np.isnan(ys[j + 1]):
            continue
        ax.plot(xs[j:j+2], ys[j:j+2], color="gray", alpha=0.30, linewidth=1)

    # ===== State points =====
    for x, y, p in zip(xs, ys, q["fes_phase"]):
        if np.isnan(y):
            continue
        ax.scatter(x, y, color=colors.get(p, "black"), s=22, zorder=3)

    # ===== Orange outlines only for aligned points =====
    for j in range(len(q)):
        if q.loc[j, "task_idx"] in highlight_task_set and np.isfinite(ys[j]):
            ax.scatter(
                xs[j], ys[j],
                s=70,
                facecolors="none",
                edgecolors="#FFA500",
                linewidth=1.2,
                alpha=0.8,
                zorder=4
            )

    # ===== J bars (these represent J itself) =====
    ax2 = ax.inset_axes([0, -0.25, 1, 0.2])
    sub_ev = ev[ev["label"] == lab]
    if len(sub_ev) > 0:
        Jmax = np.max(np.abs(sub_ev["J"])) or 1
        for _, row in sub_ev.iterrows():
            x_task = row["task_idx"]
            val = abs(row["J"]) / Jmax

            hit = q.index[q["task_idx"] == x_task].tolist()
            if len(hit) == 0:
                continue
            x_plot = hit[0] + 1
            ax2.bar(x_plot, val, color="#1E90FF", width=0.5)

    ax2.set_ylim(0, 1)
    ax2.set_yticks([])
    ax2.set_xlim(1, len(q))

    # ===== Axes formatting =====
    ax.set_title(f"{lab} (n={len(q)})", fontsize=9)
    ax.set_ylim(-0.5, len(fes_order) - 0.5)
    ax.set_yticks(range(len(fes_order)))
    ax.set_yticklabels(fes_order, fontsize=7)
    ax.set_xlim(1, len(q))
    ax.grid(alpha=0.2)

for j in range(len(labels), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "FES_FINAL_alignment_independent.png"), dpi=300)
plt.show()
plt.close()

print("Saved:", os.path.join(OUTDIR, "FES_FINAL_alignment_independent.png"))

In [ ]:
# ============================================================
# Chapter 3 Verification Script:
# 3.1 Ricci Oscillation and 3.2 Kuramoto-type Phase Coupling Test
# ------------------------------------------------------------
# Purpose:
# This script performs the numerical verification used for
# Chapter 3, Sections 3.1 and 3.2 of the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:1‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# The script evaluates two related but distinct properties of the
# neural and quantum Ricci-curvature time series:
#
#   1. Oscillatory structure of each system individually
#      - Test whether each Ricci series approximately satisfies
#            Δ²r ≈ -ω² r
#      - This corresponds to the simple harmonic oscillator
#        structure described in Section 3.1
#
#   2. Kuramoto-type phase coupling between systems
#      - Construct instantaneous phase from each Ricci series
#      - Test whether the phase difference dynamics can be
#        explained by a simple Kuramoto-type interaction model
#      - This corresponds to the explanatory test discussed in
#        Section 3.2
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression

# =========================================================
# CONFIG
# =========================================================
RICCI_DIR = "IDPC_Reproduction_ricci"
OUTDIR = "IDPC_Reproduction/Chapter3"
os.makedirs(OUTDIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(RICCI_DIR, "*_timeseries.csv")))

print("RICCI_DIR =", RICCI_DIR)
print("n_files   =", len(files))

# =========================================================
# HELPERS
# =========================================================
def safe_corr(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 3:
        return np.nan, np.nan, len(x)
    if np.nanstd(x) < 1e-12 or np.nanstd(y) < 1e-12:
        return np.nan, np.nan, len(x)
    return pearsonr(x, y)[0], spearmanr(x, y)[0], len(x)

def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def oscillator_test_one_series(r):
    r = np.asarray(r, float)
    m = np.isfinite(r)
    r = r[m]

    if len(r) < 5:
        return None

    d2r = np.diff(r, n=2)
    r_mid = r[1:-1]

    m2 = np.isfinite(d2r) & np.isfinite(r_mid)
    d2r = d2r[m2]
    r_mid = r_mid[m2]

    if len(d2r) < 3:
        return None

    rp, rs, n = safe_corr(d2r, -r_mid)

    # Δ²r ≈ -ω² r を原点通過回帰で評価
    X = r_mid.reshape(-1, 1)
    y = -d2r
    model = LinearRegression(fit_intercept=False)
    model.fit(X, y)

    omega2 = float(model.coef_[0])
    r2 = float(model.score(X, y))

    return {
        "pearson": rp,
        "spearman": rs,
        "n": n,
        "omega2": omega2,
        "r2": r2,
        "r_mid": r_mid,
        "d2r": d2r,
    }

def kuramoto_test_one_pair(rE, rQ):
    rE = np.asarray(rE, float)
    rQ = np.asarray(rQ, float)

    m = np.isfinite(rE) & np.isfinite(rQ)
    rE = rE[m]
    rQ = rQ[m]

    if len(rE) < 10:
        return None

    drE = np.gradient(rE)
    drQ = np.gradient(rQ)

    psiE = np.angle(rE + 1j * drE)
    psiQ = np.angle(rQ + 1j * drQ)

    dpsi = np.angle(np.exp(1j * (psiE - psiQ)))
    dpsi_dt = np.gradient(dpsi)

    sin_term = np.sin(dpsi)

    m2 = np.isfinite(dpsi_dt) & np.isfinite(sin_term)
    if m2.sum() < 10:
        return None

    X = sin_term[m2].reshape(-1, 1)
    y = dpsi_dt[m2]

    model = LinearRegression().fit(X, y)
    r2 = float(model.score(X, y))
    delta_omega = float(model.intercept_)
    K_est = float(-model.coef_[0] / 2.0)

    return {
        "kuramoto_r2": r2,
        "Delta_omega": delta_omega,
        "K_est": K_est,
        "n": int(m2.sum()),
        "dpsi": dpsi[m2],
    }



# =========================================================
# MAIN
# =========================================================
osc_rows = []
kur_rows = []
phase_rows = []

all_E_mid, all_E_d2 = [], []
all_Q_mid, all_Q_d2 = [], []
all_dpsi = []

for fp in files:
    label = os.path.basename(fp).replace("_timeseries.csv", "")
    df = pd.read_csv(fp)

    # 列名候補
    e_col = next((c for c in ["E_Ricci", "E", "E_true"] if c in df.columns), None)
    q_col = next((c for c in ["Q_Ricci_affine", "Q_Ricci", "Q", "Q_true"] if c in df.columns), None)

    if e_col is None or q_col is None:
        print(f"SKIP {label}: missing Ricci columns")
        continue

    rE = pd.to_numeric(df[e_col], errors="coerce").to_numpy(float)
    rQ = pd.to_numeric(df[q_col], errors="coerce").to_numpy(float)

    # -------------------------
    # Oscillator test: E
    # -------------------------
    resE = oscillator_test_one_series(rE)
    if resE is not None:
        osc_rows.append({
            "label": label,
            "series": "E",
            "n": resE["n"],
            "pearson": resE["pearson"],
            "spearman": resE["spearman"],
            "omega2": resE["omega2"],
            "r2": resE["r2"],
        })
        all_E_mid.append(resE["r_mid"])
        all_E_d2.append(resE["d2r"])

    # -------------------------
    # Oscillator test: Q
    # -------------------------
    resQ = oscillator_test_one_series(rQ)
    if resQ is not None:
        osc_rows.append({
            "label": label,
            "series": "Q",
            "n": resQ["n"],
            "pearson": resQ["pearson"],
            "spearman": resQ["spearman"],
            "omega2": resQ["omega2"],
            "r2": resQ["r2"],
        })
        all_Q_mid.append(resQ["r_mid"])
        all_Q_d2.append(resQ["d2r"])

 
    # -------------------------
    # Kuramoto test
    # -------------------------
    kt = kuramoto_test_one_pair(rE, rQ)
    if kt is not None:
        kur_rows.append({
            "label": label,
            "kuramoto_r2": kt["kuramoto_r2"],
            "Delta_omega": kt["Delta_omega"],
            "K_est": kt["K_est"],
            "n": kt["n"],
        })

osc_df = pd.DataFrame(osc_rows)
kur_df = pd.DataFrame(kur_rows)

# =========================================================
# POOLED OSCILLATOR SUMMARY
# =========================================================
def pooled_osc_summary(name, mids, d2s):
    if len(mids) == 0:
        return None
    x = np.concatenate(mids)
    d2 = np.concatenate(d2s)

    rp, rs, n = safe_corr(d2, -x)

    model = LinearRegression(fit_intercept=False)
    model.fit(x.reshape(-1, 1), -d2)
    omega2 = float(model.coef_[0])
    r2 = float(model.score(x.reshape(-1, 1), -d2))

    return {
        "series": name,
        "n": n,
        "pearson": rp,
        "spearman": rs,
        "omega2": omega2,
        "r2": r2,
    }

pooled_E = pooled_osc_summary("E_pooled", all_E_mid, all_E_d2)
pooled_Q = pooled_osc_summary("Q_pooled", all_Q_mid, all_Q_d2)



# =========================================================
# GLOBAL KURAMOTO SUMMARY
# =========================================================
if len(kur_df):
    mean_r2 = float(kur_df["kuramoto_r2"].mean())
    median_r2 = float(kur_df["kuramoto_r2"].median())
    mean_K = float(kur_df["K_est"].mean())
else:
    mean_r2 = np.nan
    median_r2 = np.nan
    mean_K = np.nan

# =========================================================
# PRINT
# =========================================================
print("\n=== Oscillator test: Δ²r = -ω²r ===")
if pooled_E is not None:
    print("E pooled :", pooled_E)
if pooled_Q is not None:
    print("Q pooled :", pooled_Q)

if len(osc_df):
    print("\nPer-session oscillator summary (head):")
    print(osc_df.head(12).to_string(index=False))


print("\n=== Kuramoto coupling test ===")
if len(kur_df):
    print(kur_df.sort_values("kuramoto_r2", ascending=False).to_string(index=False))
    print("\nGLOBAL")
    print("mean R2 =", mean_r2)
    print("median R2 =", median_r2)
    print("mean K =", mean_K)
else:
    print("No valid sessions.")

# =========================================================
# SAVE
# =========================================================
osc_df.to_csv(os.path.join(OUTDIR, "oscillator_test_per_session_latest_riccisync.csv"), index=False)
kur_df.to_csv(os.path.join(OUTDIR, "kuramoto_test_per_session_latest_riccisync.csv"), index=False)

summary_rows = []
if pooled_E is not None:
    summary_rows.append(pooled_E)
if pooled_Q is not None:
    summary_rows.append(pooled_Q)


summary_rows.append({
    "series": "kuramoto_global",
    "n": len(kur_df),
    "pearson": np.nan,
    "spearman": np.nan,
    "omega2": np.nan,
    "r2": mean_r2,
    "psi_lock_R": np.nan,
    "circ_mean_deg": np.nan,
    "mean_abs_dpsi_deg": np.nan,
    "median_r2": median_r2,
    "mean_K": mean_K,
})

pd.DataFrame(summary_rows).to_csv(
    os.path.join(OUTDIR, "latest_riccisync_global_verification_summary.csv"),
    index=False
)

print("\nSaved:")
print(" -", os.path.join(OUTDIR, "oscillator_test_per_session_latest_riccisync.csv"))
print(" -", os.path.join(OUTDIR, "kuramoto_test_per_session_latest_riccisync.csv"))
print(" -", os.path.join(OUTDIR, "latest_riccisync_global_verification_summary.csv"))

In [ ]:
# ============================================================
# Chapter 3 Figure 9 Script:
# 3.2 Dynamic Phase Synchronization of Ricci Curvature
# ------------------------------------------------------------
# Purpose:
# This script generates the quantitative summary and figure for
# Section 3.2 of the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:0‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# The goal is to evaluate phase synchronization between the
# neural Ricci curvature series and the affine-aligned quantum
# Ricci curvature series across sessions.
#
# Specifically, the script:
#   - estimates an effective oscillation scale ω for each series
#   - constructs instantaneous Ricci phase ψ from (r, dr/dt)
#   - computes the phase difference:
#         Δψ = ψ_E - ψ_Q
#   - quantifies session-wise phase-lock strength:
#         R = | <exp(iΔψ)> |
#   - summarizes the pooled phase-difference distribution
#
# This corresponds to the empirical claim in Section 3.2 that:
#   - phase differences are concentrated near zero with finite spread
#   - session-wise phase-lock values remain consistently structured
#   - the systems exhibit partial phase alignment rather than
#     trivial perfect synchronization
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RICCI_DIR = "IDPC_Reproduction_ricci"
OUTCSV = "IDPC_Reproduction/Chapter3/ricci_phase_sync_summary.csv"
OUTPNG = "IDPC_Reproduction/Chapter3/Fig_ricci_phase_sync.png"


os.makedirs("IDPC_Reproduction/Chapter3", exist_ok=True)

# =====================================================
# helpers
# =====================================================
def wrap_rad(x):
    return np.angle(np.exp(1j * np.asarray(x, float)))

def wrap_deg(x):
    x = np.asarray(x, float)
    return ((x + 180.0) % 360.0) - 180.0

def estimate_omega(r):
    r = np.asarray(r, float)
    d2 = np.diff(r, 2)
    r_mid = r[1:-1]
    m = np.isfinite(d2) & np.isfinite(r_mid) & (np.abs(r_mid) > 1e-12)
    if np.sum(m) < 5:
        return np.nan
    w2 = np.nanmean(-d2[m] / r_mid[m])
    return np.sqrt(abs(w2)) if np.isfinite(w2) else np.nan

def ricci_phase(r, omega):
    r = np.asarray(r, float)
    dr = np.diff(r, prepend=np.nan)
    psi = np.arctan2(dr / (omega + 1e-12), r)
    return wrap_rad(psi)

def circ_mean(x):
    return np.angle(np.mean(np.exp(1j * np.asarray(x, float))))

# =====================================================
# pooled Δψ and session-wise lock
# =====================================================
all_dpsi = []
rows = []

files = sorted([f for f in os.listdir(RICCI_DIR) if f.endswith("_timeseries.csv")])

for f in files:
    label = f.replace("_timeseries.csv", "")
    df = pd.read_csv(os.path.join(RICCI_DIR, f))

    if "E_Ricci" not in df.columns or "Q_Ricci_affine" not in df.columns:
        continue

    rE = pd.to_numeric(df["E_Ricci"], errors="coerce").to_numpy(float)
    rQ = pd.to_numeric(df["Q_Ricci_affine"], errors="coerce").to_numpy(float)

    n = min(len(rE), len(rQ))
    if n < 20:
        continue

    rE = rE[:n]
    rQ = rQ[:n]

    wE = estimate_omega(rE)
    wQ = estimate_omega(rQ)
    if not np.isfinite(wE) or not np.isfinite(wQ):
        continue

    psiE = ricci_phase(rE, wE)
    psiQ = ricci_phase(rQ, wQ)

    dpsi = wrap_rad(psiE - psiQ)
    m = np.isfinite(dpsi)
    dpsi = dpsi[m]

    if len(dpsi) < 10:
        continue

    R = np.abs(np.mean(np.exp(1j * dpsi)))
    mean_abs = np.mean(np.abs(dpsi))
    cmean = circ_mean(dpsi)

    all_dpsi.extend(dpsi.tolist())

    rows.append({
        "label": label,
        "n_points": len(dpsi),
        "psi_lock_R": float(R),
        "circ_mean_rad": float(cmean),
        "circ_mean_deg": float(np.degrees(cmean)),
        "mean_abs_dpsi_rad": float(mean_abs),
        "mean_abs_dpsi_deg": float(np.degrees(mean_abs)),
    })

res = pd.DataFrame(rows)
if len(res) == 0:
    raise ValueError("No valid phase data were created.")

res = res.sort_values("psi_lock_R", ascending=False).reset_index(drop=True)
res.to_csv(OUTCSV, index=False)

all_dpsi = np.asarray(all_dpsi, float)
all_dpsi_deg = np.degrees(all_dpsi)

pooled_R = np.abs(np.mean(np.exp(1j * all_dpsi)))
pooled_cmean = circ_mean(all_dpsi)
pooled_cmean_deg = np.degrees(pooled_cmean)
pooled_mean_abs_deg = np.degrees(np.mean(np.abs(all_dpsi)))

print("\n=== Ricci phase synchronization summary ===\n")
print(f"N pooled points = {len(all_dpsi)}")
print(f"pooled psi-lock R = {pooled_R:.4f}")
print(f"pooled circular mean (deg) = {pooled_cmean_deg:.3f}")
print(f"pooled mean |Δpsi| (deg) = {pooled_mean_abs_deg:.3f}")
print()
print(res.to_string(index=False))

# =====================================================
# figure
# =====================================================
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.8))

# Panel A: pooled Δψ histogram
bins = np.linspace(-180, 180, 37)
axs[0].hist(all_dpsi_deg, bins=bins)
axs[0].axvline(0, color="black", linewidth=1)
axs[0].axvline(pooled_cmean_deg, linestyle="--", linewidth=1.5)

axs[0].set_xlabel(r"$\Delta \psi$ (deg)")
axs[0].set_ylabel("Count")

txt = (
    f"N = {len(all_dpsi)}\n"
    f"R = {pooled_R:.3f}\n"
    f"circ. mean = {pooled_cmean_deg:.1f}°\n"
    f"mean |Δψ| = {pooled_mean_abs_deg:.1f}°"
)
axs[0].text(
    0.98, 0.98, txt,
    transform=axs[0].transAxes,
    ha="right", va="top",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

# Panel B: session-wise ψ-lock
x = np.arange(len(res))
axs[1].plot(x, res["psi_lock_R"].values, marker="o", linewidth=1.8)
axs[1].axhline(res["psi_lock_R"].mean(), linestyle="--", linewidth=1.5)

axs[1].set_xticks(x)
axs[1].set_xticklabels(res["label"], rotation=90)
axs[1].set_xlabel("Session")
axs[1].set_ylabel(r"Phase-lock strength $R$")

plt.tight_layout()
plt.savefig(OUTPNG, dpi=300, bbox_inches="tight")
plt.show()

plt.close()

print("\nSaved:")
print(OUTCSV)
print(OUTPNG)

In [ ]:
# ============================================================
# Chapter 3 Figure 10 Script:
# 3.3 Structural Phase Residual and Contraction
# ------------------------------------------------------------
# Purpose:
# This script generates the pooled verification and figure for
# Section 3.3 of the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:0‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# The goal is to test whether the structural phase difference
# between neural and quantum Ricci dynamics contracts toward
# a 72-degree periodic lattice.
#
# Specifically, the script:
#   - computes Ricci phase for neural and affine quantum series
#   - defines the structural phase difference:
#         Δθ_t = θ_E(t) - θ_Q(t)
#   - converts this phase difference into a 72° lattice residual:
#         ε_t ∈ [-36°, 36°)
#   - evaluates whether the residual exhibits restoring dynamics:
#         Δε_t ≈ -λ ε_t
#
# This corresponds to the empirical claim in Section 3.3 that:
#   - structural phase differences do not drift freely
#   - instead they are pulled back toward a periodic attractor
#   - the observed relation is contractive rather than diffusive
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RICCI_DIR = "IDPC_Reproduction_ricci"
OUTCSV = "IDPC_Reproduction/Chapter3/ricci_eps72_restoring_test.csv"
OUTPNG = "IDPC_Reproduction/Chapter3/Fig_ricci_eps72_restoring_final.png"

os.makedirs("IDPC_Reproduction", exist_ok=True)

# =====================================================
# helpers
# =====================================================
def wrap_rad(x):
    return np.angle(np.exp(1j * np.asarray(x, float)))

def wrap_deg(x):
    x = np.asarray(x, float)
    return ((x + 180.0) % 360.0) - 180.0

def eps72_deg(theta_deg):
    """
    72° periodic lattice residual in [-36, 36)
    """
    theta_deg = wrap_deg(theta_deg)
    return ((theta_deg + 36.0) % 72.0) - 36.0

def estimate_omega(r):
    r = np.asarray(r, float)
    d2 = np.diff(r, 2)
    r_mid = r[1:-1]
    m = np.isfinite(d2) & np.isfinite(r_mid) & (np.abs(r_mid) > 1e-12)
    if np.sum(m) < 5:
        return np.nan
    w2 = np.nanmean(-d2[m] / r_mid[m])
    return np.sqrt(abs(w2)) if np.isfinite(w2) else np.nan

def ricci_phase(r, omega):
    r = np.asarray(r, float)
    dr = np.diff(r, prepend=np.nan)
    psi = np.arctan2(dr / (omega + 1e-12), r)
    return wrap_rad(psi)

# =====================================================
# pooled data build
# =====================================================
rows = []

files = sorted([f for f in os.listdir(RICCI_DIR) if f.endswith("_timeseries.csv")])

for f in files:
    label = f.replace("_timeseries.csv", "")
    df = pd.read_csv(os.path.join(RICCI_DIR, f))

    if "E_Ricci" not in df.columns or "Q_Ricci_affine" not in df.columns:
        continue

    rE = pd.to_numeric(df["E_Ricci"], errors="coerce").to_numpy(float)
    rQ = pd.to_numeric(df["Q_Ricci_affine"], errors="coerce").to_numpy(float)

    n = min(len(rE), len(rQ))
    if n < 20:
        continue

    rE = rE[:n]
    rQ = rQ[:n]

    wE = estimate_omega(rE)
    wQ = estimate_omega(rQ)
    if not np.isfinite(wE) or not np.isfinite(wQ):
        continue

    psiE = ricci_phase(rE, wE)
    psiQ = ricci_phase(rQ, wQ)

    # Ricci-transformed structural phase difference
    theta_deg = np.degrees(wrap_rad(psiE - psiQ))

    # 72° periodic residual
    eps = eps72_deg(theta_deg)

    eps_t = eps[:-1]
    deps = eps[1:] - eps[:-1]
    deps = wrap_deg(deps)

    m = np.isfinite(eps_t) & np.isfinite(deps)
    eps_t = eps_t[m]
    deps = deps[m]

    for e, d in zip(eps_t, deps):
        rows.append({
            "label": label,
            "eps72_deg": float(e),
            "deps72_deg": float(d),
            "restore": int(e * d < 0)
        })

dat = pd.DataFrame(rows)
dat.to_csv(OUTCSV, index=False)

if len(dat) == 0:
    raise ValueError("No valid pooled eps72 data were created.")

# =====================================================
# summary statistics
# =====================================================
x = dat["eps72_deg"].to_numpy(float)
y = dat["deps72_deg"].to_numpy(float)

slope = np.sum(x * y) / np.sum(x * x)
restore_rate = dat["restore"].mean()

# binned drift
bins = np.linspace(-36, 36, 13)
dat["eps_bin"] = pd.cut(dat["eps72_deg"], bins=bins, include_lowest=True)

b = dat.groupby("eps_bin", observed=False).agg(
    eps_mean=("eps72_deg", "mean"),
    deps_mean=("deps72_deg", "mean"),
    n=("deps72_deg", "size")
).reset_index()

print("\n=== 72° restoring test ===\n")
print(f"N pooled = {len(dat)}")
print(f"slope in Δeps = slope * eps : {slope:.4f}")
print(f"restore rate P(eps*Δeps < 0) = {restore_rate:.4f}")
print("\nBinned drift:")
print(b.to_string(index=False))

# =====================================================
# figure
# =====================================================
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.8))

# Panel A: pooled regression
axs[0].scatter(x, y, s=10, alpha=0.25)
xx = np.linspace(-36, 36, 200)
axs[0].plot(xx, slope * xx, linewidth=2)
axs[0].axhline(0, color="black", linewidth=1)
axs[0].axvline(0, color="black", linewidth=1)
axs[0].set_xlabel(r"$\varepsilon_t$ (deg)")
axs[0].set_ylabel(r"$\Delta \varepsilon_t$ (deg)")

txt = (
    f"N = {len(dat)}\n"
    f"slope = {slope:.3f}\n"
    f"restore rate = {restore_rate:.3f}"
)
axs[0].text(
    0.98, 0.98, txt,
    transform=axs[0].transAxes,
    ha="right", va="top",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9)
)

# Panel B: binned drift
axs[1].plot(b["eps_mean"], b["deps_mean"], marker="o", linewidth=2)
axs[1].axhline(0, color="black", linewidth=1)
axs[1].axvline(0, color="black", linewidth=1)
axs[1].set_xlabel(r"mean $\varepsilon_t$ in bin (deg)")
axs[1].set_ylabel(r"mean $\Delta \varepsilon_t$ (deg)")

plt.tight_layout()
plt.savefig(OUTPNG, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("\nSaved:")
print(OUTCSV)
print(OUTPNG)

In [ ]:
# ============================================================
# Chapter 3 Verification Script:
# 3.5 Significance of State Transitions
# ------------------------------------------------------------
# Purpose:
# This script performs the permutation-based significance test
# for the transition structure described in Section 3.5 of the
# paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:0‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# The goal is to test whether the observed FES state-transition
# structure can be explained by random ordering of states within
# sessions, or whether it reflects a genuine structural constraint.
#
# Specifically, the script:
#   - constructs the observed transition-count matrix
#   - converts it to a row-normalized transition-probability matrix
#   - computes a global structure score measuring deviation from
#     a uniform transition model
#   - evaluates specific directed transitions of interest
#   - compares all observed values against a within-session
#     permutation null distribution
#
# This corresponds to the empirical claim in Section 3.5 that:
#   - the transition structure is not random
#   - the overall transition organization significantly exceeds
#     permutation null expectations
#   - certain directed transitions show elevated probabilities
#     relative to the null baseline
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================
# CONFIG
# =========================================
INDIR = "IDPC_Reproduction"
CSV = os.path.join(INDIR, "event_level_with_fes_phase_TRUE_RICCI.csv")
OUTDIR = os.path.join(INDIR, "Chapter3/MarkovTest")
os.makedirs(OUTDIR, exist_ok=True)

N_PERM = 5000
SEED = 42

fes_order = ["Activation", "Challenge", "Surprise", "SelfGrowth", "CoCreation"]

# 注目遷移
target_edges = [
    ("Activation", "CoCreation"),
    ("Challenge", "Surprise"),
    ("Challenge", "CoCreation"),
    ("Surprise", "CoCreation"),
    ("CoCreation", "Surprise"),
    ("SelfGrowth", "Surprise"),
]

# =========================================
# HELPERS
# =========================================
def build_transition_counts(df, states):
    mat = pd.DataFrame(0, index=states, columns=states, dtype=float)

    for lab, g in df.groupby("label"):
        g = g.sort_values("task_idx")
        phases = g["fes_phase"].tolist()

        for i in range(len(phases) - 1):
            a = phases[i]
            b = phases[i + 1]
            if pd.isna(a) or pd.isna(b):
                continue
            if a not in states or b not in states:
                continue
            mat.loc[a, b] += 1
    return mat

def row_normalize(mat):
    out = mat.copy().astype(float)
    rs = out.sum(axis=1)
    for i in out.index:
        if rs.loc[i] > 0:
            out.loc[i, :] = out.loc[i, :] / rs.loc[i]
        else:
            out.loc[i, :] = np.nan
    return out

def structure_score_prob(mat_prob, uniform=0.2):
    vals = mat_prob.to_numpy(float)
    mask = np.isfinite(vals)
    return float(np.sum((vals[mask] - uniform)**2))

def edge_score(mat_prob, a, b):
    v = mat_prob.loc[a, b]
    return float(v) if np.isfinite(v) else np.nan

def permute_within_session(df, rng):
    pieces = []
    for lab, g in df.groupby("label"):
        g = g.sort_values("task_idx").copy()
        phases = g["fes_phase"].to_numpy(object)
        idx = np.arange(len(phases))
        rng.shuffle(idx)
        g["fes_phase"] = phases[idx]
        pieces.append(g)
    return pd.concat(pieces, axis=0).reset_index(drop=True)

def permutation_pvalue(obs, null):
    null = np.asarray(null, float)
    null = null[np.isfinite(null)]
    if len(null) == 0:
        return np.nan
    # 右片側
    return float((np.sum(null >= obs) + 1) / (len(null) + 1))

# =========================================
# LOAD
# =========================================
df = pd.read_csv(CSV)

if not {"label", "task_idx", "fes_phase"}.issubset(df.columns):
    raise KeyError(f"Need columns label, task_idx, fes_phase in {CSV}")

df = df.dropna(subset=["label", "task_idx", "fes_phase"]).copy()
df["task_idx"] = pd.to_numeric(df["task_idx"], errors="coerce")
df = df.dropna(subset=["task_idx"]).copy()
df["task_idx"] = df["task_idx"].astype(int)

# =========================================
# OBSERVED
# =========================================
obs_counts = build_transition_counts(df, fes_order)
obs_prob = row_normalize(obs_counts)
obs_structure = structure_score_prob(obs_prob, uniform=1/len(fes_order))

obs_edge_scores = {}
for a, b in target_edges:
    obs_edge_scores[f"{a}->{b}"] = edge_score(obs_prob, a, b)

obs_counts.to_csv(os.path.join(OUTDIR, "transition_counts_observed.csv"))
obs_prob.to_csv(os.path.join(OUTDIR, "transition_prob_observed.csv"))

print("\n=== OBSERVED TRANSITION COUNTS ===")
print(obs_counts)

print("\n=== OBSERVED TRANSITION PROBABILITY ===")
print(obs_prob)

print("\n=== OBSERVED STRUCTURE SCORE ===")
print(obs_structure)

print("\n=== OBSERVED TARGET EDGES ===")
for k, v in obs_edge_scores.items():
    print(k, "=", v)

# =========================================
# PERMUTATION
# =========================================
rng = np.random.default_rng(SEED)

null_structure = []
null_edges = {f"{a}->{b}": [] for a, b in target_edges}

for r in range(N_PERM):
    dfp = permute_within_session(df, rng)

    cnt = build_transition_counts(dfp, fes_order)
    prb = row_normalize(cnt)

    null_structure.append(structure_score_prob(prb, uniform=1/len(fes_order)))

    for a, b in target_edges:
        null_edges[f"{a}->{b}"].append(edge_score(prb, a, b))

    if (r + 1) % 500 == 0:
        print(f"perm {r+1}/{N_PERM}")

null_structure = np.asarray(null_structure, float)

# =========================================
# PVALUES
# =========================================
rows = []

rows.append({
    "metric": "structure_score",
    "observed": obs_structure,
    "null_mean": float(np.nanmean(null_structure)),
    "null_sd": float(np.nanstd(null_structure)),
    "null_q95": float(np.nanquantile(null_structure, 0.95)),
    "p_perm": permutation_pvalue(obs_structure, null_structure),
})

for k, obs in obs_edge_scores.items():
    arr = np.asarray(null_edges[k], float)
    rows.append({
        "metric": k,
        "observed": obs,
        "null_mean": float(np.nanmean(arr)),
        "null_sd": float(np.nanstd(arr)),
        "null_q95": float(np.nanquantile(arr, 0.95)),
        "p_perm": permutation_pvalue(obs, arr),
    })

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUTDIR, "permutation_results.csv"), index=False)

print("\n=== PERMUTATION RESULTS ===")
print(res.to_string(index=False))

# =========================================
# PLOTS
# =========================================
# 1) structure score histogram
plt.figure(figsize=(6,4))
plt.hist(null_structure, bins=40, alpha=0.8)
plt.axvline(obs_structure, linewidth=2)
plt.title("Permutation null: structure score")
plt.xlabel("structure score")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "perm_structure_score_hist.png"), dpi=220)
plt.close()

# 2) edge histograms
for k, obs in obs_edge_scores.items():
    arr = np.asarray(null_edges[k], float)
    arr = arr[np.isfinite(arr)]

    plt.figure(figsize=(6,4))
    plt.hist(arr, bins=40, alpha=0.8)
    plt.axvline(obs, linewidth=2)
    plt.title(f"Permutation null: {k}")
    plt.xlabel("transition probability")
    plt.ylabel("count")
    plt.tight_layout()
    safe_name = k.replace("->", "_to_")
    plt.savefig(os.path.join(OUTDIR, f"perm_{safe_name}_hist.png"), dpi=220)
    plt.close()

# 3) heatmap observed
plt.figure(figsize=(6,5))
plt.imshow(obs_prob.values, cmap="viridis")
plt.xticks(range(len(fes_order)), fes_order, rotation=45)
plt.yticks(range(len(fes_order)), fes_order)
plt.colorbar(label="transition probability")
plt.title("Observed transition matrix")

for i in range(len(fes_order)):
    for j in range(len(fes_order)):
        v = obs_prob.values[i, j]
        if np.isfinite(v):
            plt.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "observed_transition_heatmap.png"), dpi=220)
plt.close()

print("\nSaved to:", OUTDIR)
print(" - transition_counts_observed.csv")
print(" - transition_prob_observed.csv")
print(" - permutation_results.csv")
print(" - perm_structure_score_hist.png")
print(" - observed_transition_heatmap.png")
for a, b in target_edges:
    print(" -", f"perm_{a}_to_{b}_hist.png")

In [ ]:
# ============================================================
# Chapter 3 Figure 11 Script:
# 3.6 Boundary Events and Boundary Impulse Law
# ------------------------------------------------------------
# Purpose:
# This script generates the pooled verification plot for
# Section 3.6 of the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:0‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# The goal is to test whether boundary events obey a continuous
# quantitative law relating the boundary coordinate change Δh
# and the boundary impulse J.
#
# Specifically, the script:
#   - loads pooled boundary-event data across sessions
#   - separates entry and exit events by the sign of Δh
#   - fits the origin-through relation:
#         J ≈ α · Δh
#   - evaluates goodness of fit using R²
#   - visualizes both event classes together with the fitted law
#
# This corresponds to the empirical claim in Section 3.6 that:
#   - state transitions concentrate at boundary events
#   - boundary impulses are not arbitrary discrete triggers
#   - instead, they follow a continuous structural constraint
#     across both entry and exit events
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


OUTDIR = "IDPC_Reproduction/Figures"
os.makedirs(OUTDIR, exist_ok=True)

# ==========
# Load pooled event data
# ==========
df = pd.read_csv("IDPC_Reproduction/J_dh_kappa_pooled_v2.csv")

dh = df["dh"].to_numpy(float)
J  = df["J"].to_numpy(float)

print("n events:", len(dh))

# ==========
# enter / exit mask
# ==========
enter_mask = dh > 0
exit_mask  = dh < 0

print("enter:", np.sum(enter_mask),
      "exit:", np.sum(exit_mask))

# ==========
# Origin-through fit: J = alpha * dh
# ==========
alpha_hat = np.sum(J * dh) / np.sum(dh * dh)

# R²
J_pred = alpha_hat * dh
R2 = 1 - np.sum((J - J_pred)**2) / np.sum((J - np.mean(J))**2)

print("alpha_hat:", alpha_hat)
print("R2:", R2)

# ==========
# Plot
# ==========
plt.figure(figsize=(7,6))

# scatter
plt.scatter(dh[enter_mask], J[enter_mask],
            color="blue", alpha=0.7, label="enter (h−→+)")
plt.scatter(dh[exit_mask], J[exit_mask],
            color="orange", alpha=0.7, label="exit (h+→−)")

# theory line
x_line = np.linspace(dh.min()*1.1, dh.max()*1.1, 300)
plt.plot(x_line, alpha_hat * x_line,
         color="black", linewidth=2,
         label=f"Fit: J = {alpha_hat:.2f} · Δh")

plt.axhline(0, color="gray")
plt.axvline(0, color="gray")

plt.xlabel("Δh = h₁ − h₀")
plt.ylabel("J = a_post − a_pre")
plt.title("Boundary Impulse Law (26 sessions pooled)")
plt.legend()
plt.tight_layout()

plt.savefig("IDPC_Reproduction/Chapter3/IDPE_BI_boundary_impulse_pooled.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# Chapter 5 Figure 12 and 13 Script:
# 5.4 Potential–State Correspondence and
# 5.5 Distribution of High-Reconstruction Regions
# ------------------------------------------------------------
# Purpose:
# This script generates Figures 12 and 13 for Chapter 5 of the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:0‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# The goal is to demonstrate that:
#   - the intersection variable φ induces a multistable potential
#   - discrete states are not uniformly distributed, but are
#     constrained by potential wells
#   - reconstruction performance is not global, but selectively
#     realized under specific (well × state) conditions
#
# Figure 12:
#   A. Empirical potential U(φ)
#   B. State occupancy per potential well
#   C. Reconstruction performance (well × state)
#   D. Conceptual generative structure
#
# Figure 13:
#   Distribution of high-reconstruction points on the φ coordinate,
#   showing localization in specific potential regions
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter1d
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler

# ----------------------------
# Configuration
# ----------------------------
INDIR = "IDPC_Reproduction"
EVENT_FILE = os.path.join(INDIR, "event_level_with_fes_phase_TRUE_RICCI.csv")

OUTDIR = os.path.join("IDPC_Reproduction", "Chapter5")
os.makedirs(OUTDIR, exist_ok=True)


N_BINS = 40
SMOOTH_SIGMA = 1.2
EPS = 1e-8
MIN_GROUP_N = 5

# ----------------------------
# Load event-level data
# ----------------------------
df = pd.read_csv(EVENT_FILE)
df = df.sort_values(["label", "task_idx"]).reset_index(drop=True)

required_cols = ["label", "task_idx", "phi", "dphi", "E", "Q", "fes_phase"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

for c in ["phi", "dphi", "E", "Q"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ----------------------------
# Session-wise normalization
# phi_norm = phi / session_std(phi)
# dphi_norm = dphi / session_std(dphi)
# ----------------------------
norm_parts = []

for label, g in df.groupby("label"):
    g = g.sort_values("task_idx").reset_index(drop=True).copy()

    phi_std = np.nanstd(g["phi"].values)
    dphi_std = np.nanstd(g["dphi"].values)

    if not np.isfinite(phi_std) or phi_std < 1e-12:
        phi_std = 1.0
    if not np.isfinite(dphi_std) or dphi_std < 1e-12:
        dphi_std = 1.0

    g["phi_norm"] = g["phi"] / phi_std
    g["dphi_norm"] = g["dphi"] / dphi_std
    norm_parts.append(g)

df = pd.concat(norm_parts, ignore_index=True)
df = df.sort_values(["label", "task_idx"]).reset_index(drop=True)

# ----------------------------
# Build training rows
# The next state is restricted to:
#   - CoCreation
#   - Surprise
# ----------------------------
rows = []

for label, g in df.groupby("label"):
    g = g.sort_values("task_idx").reset_index(drop=True)

    for i in range(1, len(g) - 1):
        next_state = g.loc[i + 1, "fes_phase"]
        if next_state not in ["CoCreation", "Surprise"]:
            continue

        vals = [
            g.loc[i, "phi_norm"], g.loc[i, "dphi_norm"],
            g.loc[i, "E"], g.loc[i, "Q"],
            g.loc[i - 1, "E"], g.loc[i - 1, "Q"],
            g.loc[i + 1, "E"], g.loc[i + 1, "Q"]
        ]
        if not np.all(np.isfinite(vals)):
            continue

        E_t = g.loc[i, "E"]
        Q_t = g.loc[i, "Q"]
        E_tm1 = g.loc[i - 1, "E"]
        Q_tm1 = g.loc[i - 1, "Q"]

        rows.append({
            "label": label,
            "task_idx_t": g.loc[i, "task_idx"],
            "phi_norm_t": g.loc[i, "phi_norm"],
            "dphi_norm_t": g.loc[i, "dphi_norm"],
            "E_t": E_t,
            "Q_t": Q_t,
            "E_tm1": E_tm1,
            "Q_tm1": Q_tm1,
            "dE_t": E_t - E_tm1,
            "dQ_t": Q_t - Q_tm1,
            "E_next": g.loc[i + 1, "E"],
            "Q_next": g.loc[i + 1, "Q"],
            "state_next": next_state,
            "y_state": 1 if next_state == "Surprise" else 0
        })

train = pd.DataFrame(rows).dropna().reset_index(drop=True)
if len(train) == 0:
    raise ValueError("No valid training rows were created.")

# ----------------------------
# Train state classifier
# State is predicted from (phi_norm, dphi_norm)
# ----------------------------
X_state = train[["phi_norm_t", "dphi_norm_t"]].values
y_state = train["y_state"].values

state_scaler = StandardScaler()
X_state_scaled = state_scaler.fit_transform(X_state)

state_clf = LogisticRegression(max_iter=1000)
state_clf.fit(X_state_scaled, y_state)

# ----------------------------
# Train state-specific dynamics
# AR2 + delta model for E and Q
# ----------------------------
dyn_models = {}

for state_name in ["CoCreation", "Surprise"]:
    sub = train[train["state_next"] == state_name].copy()
    if len(sub) < MIN_GROUP_N:
        raise ValueError(f"Too few samples for dynamics model: {state_name}")

    XE = sub[["E_t", "E_tm1", "dE_t"]].values
    yE = sub["E_next"].values
    XQ = sub[["Q_t", "Q_tm1", "dQ_t"]].values
    yQ = sub["Q_next"].values

    dyn_models[state_name] = {
        "E_model": LinearRegression().fit(XE, yE),
        "Q_model": LinearRegression().fit(XQ, yQ)
    }

# ----------------------------
# Build point-level table for figure generation
# Each point includes:
#   - phi_norm
#   - predicted state
#   - next-step prediction errors
#   - reconstruction score
# ----------------------------
points = []

for label, g in df.groupby("label"):
    g = g.sort_values("task_idx").reset_index(drop=True)

    for i in range(1, len(g) - 1):
        vals = [
            g.loc[i, "phi_norm"], g.loc[i, "dphi_norm"],
            g.loc[i, "E"], g.loc[i, "Q"],
            g.loc[i - 1, "E"], g.loc[i - 1, "Q"],
            g.loc[i + 1, "E"], g.loc[i + 1, "Q"]
        ]
        if not np.all(np.isfinite(vals)):
            continue

        x_state = np.array([[g.loc[i, "phi_norm"], g.loc[i, "dphi_norm"]]])
        x_state_scaled = state_scaler.transform(x_state)
        p_surprise = state_clf.predict_proba(x_state_scaled)[0, 1]
        state_pred = "Surprise" if p_surprise >= 0.5 else "CoCreation"

        E_t = g.loc[i, "E"]
        Q_t = g.loc[i, "Q"]
        E_tm1 = g.loc[i - 1, "E"]
        Q_tm1 = g.loc[i - 1, "Q"]
        dE_t = E_t - E_tm1
        dQ_t = Q_t - Q_tm1

        E_hat = dyn_models[state_pred]["E_model"].predict(np.array([[E_t, E_tm1, dE_t]]))[0]
        Q_hat = dyn_models[state_pred]["Q_model"].predict(np.array([[Q_t, Q_tm1, dQ_t]]))[0]

        E_true_next = g.loc[i + 1, "E"]
        Q_true_next = g.loc[i + 1, "Q"]

        err_E = abs(E_hat - E_true_next)
        err_Q = abs(Q_hat - Q_true_next)
        score = -(err_E + err_Q)

        points.append({
            "label": label,
            "task_idx": g.loc[i, "task_idx"],
            "phi_norm": g.loc[i, "phi_norm"],
            "dphi_norm": g.loc[i, "dphi_norm"],
            "state_pred": state_pred,
            "p_surprise": p_surprise,
            "E_true_next": E_true_next,
            "Q_true_next": Q_true_next,
            "E_pred_next": E_hat,
            "Q_pred_next": Q_hat,
            "score": score
        })

points_df = pd.DataFrame(points).dropna().reset_index(drop=True)
if len(points_df) == 0:
    raise ValueError("No valid point rows were created.")

print("Training rows :", len(train))
print("Point rows    :", len(points_df))



# ----------------------------
# Estimate empirical potential U(phi)
# U(phi) = -log P(phi)
# ----------------------------
phi_vals = points_df["phi_norm"].values
hist, edges = np.histogram(phi_vals, bins=N_BINS, density=True)
centers = 0.5 * (edges[:-1] + edges[1:])

hist_smooth = gaussian_filter1d(hist.astype(float), sigma=SMOOTH_SIGMA)
prob = hist_smooth / np.sum(hist_smooth)
U = -np.log(prob + EPS)
U = U - np.min(U)

potential_df = pd.DataFrame({
    "phi_center": centers,
    "P_phi": prob,
    "U_phi": U
})
potential_df.to_csv(os.path.join(OUTDIR, "empirical_potential.csv"), index=False)

# ----------------------------
# Detect local minima as potential wells
# ----------------------------
well_idx = [
    i for i in range(1, len(U) - 1)
    if U[i] < U[i - 1] and U[i] < U[i + 1]
]

well_df = pd.DataFrame({
    "phi_well": centers[well_idx],
    "U_well": U[well_idx]
})
well_df.to_csv(os.path.join(OUTDIR, "potential_wells.csv"), index=False)

if len(well_df) == 0:
    raise ValueError("No potential wells were detected. Try adjusting N_BINS or SMOOTH_SIGMA.")

# ----------------------------
# Assign each point to the nearest well
# ----------------------------
phi_wells = well_df["phi_well"].values

assign_rows = []
for _, row in points_df.iterrows():
    phi = row["phi_norm"]
    nearest_idx = np.argmin(np.abs(phi_wells - phi))
    nearest_well = phi_wells[nearest_idx]

    assign_rows.append({
        "label": row["label"],
        "phi_norm": phi,
        "nearest_well": nearest_well,
        "state": row["state_pred"],
        "E_true": row["E_true_next"],
        "Q_true": row["Q_true_next"],
        "E_pred": row["E_pred_next"],
        "Q_pred": row["Q_pred_next"],
        "score": row["score"]
    })

assign_df = pd.DataFrame(assign_rows)

# ----------------------------
# Count occupancy by well and state
# ----------------------------
count_pivot = (
    assign_df.groupby(["nearest_well", "state"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

for col in ["CoCreation", "Surprise"]:
    if col not in count_pivot.columns:
        count_pivot[col] = 0
count_pivot = count_pivot[["CoCreation", "Surprise"]]

count_pivot_reset = count_pivot.reset_index()
count_pivot_reset.to_csv(os.path.join(OUTDIR, "well_state_counts.csv"), index=False)

# ----------------------------
# Compute reconstruction score by well x state
# Score = mean(|r_E|, |r_Q|)
# ----------------------------
heat_rows = []

for w in sorted(phi_wells):
    for st in ["CoCreation", "Surprise"]:
        sub = assign_df[
            (assign_df["nearest_well"] == w) &
            (assign_df["state"] == st)
        ].copy()

        if len(sub) < MIN_GROUP_N:
            continue
        if sub["E_true"].nunique() < 2 or sub["E_pred"].nunique() < 2:
            continue
        if sub["Q_true"].nunique() < 2 or sub["Q_pred"].nunique() < 2:
            continue

        rE = np.corrcoef(sub["E_true"], sub["E_pred"])[0, 1]
        rQ = np.corrcoef(sub["Q_true"], sub["Q_pred"])[0, 1]
        score = (abs(rE) + abs(rQ)) / 2

        if np.isfinite(score):
            heat_rows.append({
                "phi_well": w,
                "state": st,
                "score": score,
                "n": len(sub),
                "rE": rE,
                "rQ": rQ
            })

heat_df = pd.DataFrame(heat_rows)
heat_df.to_csv(os.path.join(OUTDIR, "well_state_reconstruction.csv"), index=False)

pivot = (
    heat_df.pivot(index="phi_well", columns="state", values="score")
    .sort_index()
)

for col in ["CoCreation", "Surprise"]:
    if col not in pivot.columns:
        pivot[col] = np.nan
pivot = pivot[["CoCreation", "Surprise"]]

# ----------------------------
# Figure 1: four-panel structural summary
# ----------------------------
fig = plt.figure(figsize=(10, 10))

ax1 = plt.subplot(2, 2, 1)
ax1.plot(centers, U, linewidth=2)
ax1.scatter(well_df["phi_well"], well_df["U_well"], s=80)
ax1.set_title("A. Potential U(φ)")
ax1.set_xlabel("φ_norm")
ax1.set_ylabel("U(φ)")
ax1.grid(alpha=0.3)

ax2 = plt.subplot(2, 2, 2)
labels = [f"{v:.2f}" for v in count_pivot.index]
x = np.arange(len(labels))
width = 0.35

co = count_pivot["CoCreation"].values
su = count_pivot["Surprise"].values

ax2.bar(x - width / 2, co, width, label="CoCreation", color="#FFA500")
ax2.bar(x + width / 2, su, width, label="Surprise", color="#1E90FF")
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_title("B. State occupancy by well")
ax2.set_xlabel("potential well (φ)")
ax2.set_ylabel("count")
ax2.legend()
ax2.grid(alpha=0.3, axis="y")

ax3 = plt.subplot(2, 2, 3)
im = ax3.imshow(pivot.values, cmap="viridis", aspect="auto")
ax3.set_xticks(range(len(pivot.columns)))
ax3.set_xticklabels(pivot.columns)
ax3.set_yticks(range(len(pivot.index)))
ax3.set_yticklabels([f"{v:.2f}" for v in pivot.index])
ax3.set_title("C. Reconstruction (well × state)")

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if np.isfinite(val):
            ax3.text(j, i, f"{val:.2f}", ha="center", va="center", color="white")

plt.colorbar(im, ax=ax3, label="mean |reconstruction|")

ax4 = plt.subplot(2, 2, 4)
ax4.axis("off")
concept_text = """
φ  →  U(φ)
        ↓
      well
        ↓
     state
        ↓
   dynamics
        ↓
     (E, Q)

Reconstruction occurs only when:
well × state are consistent
"""
ax4.text(0.05, 0.5, concept_text, fontsize=11, verticalalignment="center")
ax4.set_title("D. Conceptual model")

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Figure11_final_structure.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()
# ----------------------------
# Figure 2: high-reconstruction scatter
# ----------------------------
score_q = points_df["score"].quantile(0.80)
good_df = points_df[points_df["score"] >= score_q].copy()

plt.figure(figsize=(8, 5))
plt.scatter(points_df["phi_norm"], points_df["score"], alpha=0.2, label="all")
plt.scatter(good_df["phi_norm"], good_df["score"], alpha=0.8, label="top20% score")
plt.xlabel(r"$\phi_{\mathrm{norm}}$")
plt.ylabel("reconstruction score")
plt.title("High-reconstruction points on the potential coordinate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Figure12_high_score_points.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
# ============================================================
# Chapter 7 Data Preparation Script:
# Construction of the Candidate φ Dataset for Intersection-Structure Analysis
# ------------------------------------------------------------
# Purpose:
# This script prepares the base dataset used in the subsequent
# Chapter 7 analyses, including:
#   - train/test search for the best intersection variable
#   - evaluation against neural-only / quantum-only baselines
#   - permutation and temporal-shift tests
#   - Figure 14 generation
#
# The script constructs a session-wise candidate φ dataset from
# point-level structural variables derived from the *_points.csv
# files.
#
# In particular, it builds:
#   - h      : boundary-related coordinate
#   - a      : amplitude-like variable
#   - eps    : 72° periodic residual derived from h
#   - dpsi   : local phase-gradient surrogate from h
#   - phi_latent : PCA-based latent coordinate
#   - phi_clean  : rank-Gaussianized φ coordinate
#   - dphi, d2phi, dh, da, deps, ddpsi : first/second differences
#
# This dataset serves as the common input space from which
# multiple candidate intersection models are later constructed
# and compared.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

BASE_DIR = "IDPC_Reproduction"
POINT_DIR = BASE_DIR
OUT_DIR = os.path.join(BASE_DIR, "Chapter7")
os.makedirs(OUT_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(POINT_DIR, "*_points.csv")))

def safe_z(x):
    x = np.asarray(x, float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd == 0:
        sd = 1.0
    return (x - mu) / sd

def periodic_residual_deg(x, period=72.0, half=36.0):
    x = np.asarray(x, float)
    return ((x + half) % period) - half

def rank_gaussian(x):
    from statistics import NormalDist
    x = np.asarray(x, float)
    m = np.isfinite(x)
    out = np.full_like(x, np.nan, dtype=float)
    if m.sum() < 3:
        return out
    ranks = pd.Series(x[m]).rank().to_numpy() / (m.sum() + 1.0)
    nd = NormalDist()
    out[m] = [nd.inv_cdf(v) for v in ranks]
    return out

rows = []

for fp in files:
    label = os.path.basename(fp).replace("_points.csv", "")
    df = pd.read_csv(fp)

    need = ["h", "a", "valid"]
    if not all(c in df.columns for c in need):
        continue

    h = pd.to_numeric(df["h"], errors="coerce").to_numpy(float)
    a = pd.to_numeric(df["a"], errors="coerce").to_numpy(float)
    v = pd.to_numeric(df["valid"], errors="coerce").to_numpy(float) == 1

    m = np.isfinite(h) & np.isfinite(a) & v
    if np.sum(m) < 50:
        continue

    h = h[m]
    a = a[m]
    eps = periodic_residual_deg(h, period=72.0, half=36.0)
    dpsi = np.gradient(h)

    for i in range(len(h)):
        rows.append({
            "label": label,
            "idx_in_session": i,
            "h": h[i],
            "a": a[i],
            "eps": eps[i],
            "dpsi": dpsi[i]
        })

dat = pd.DataFrame(rows).sort_values(["label", "idx_in_session"]).reset_index(drop=True)

X = dat[["h", "a", "eps", "dpsi"]].to_numpy(float)
X = np.column_stack([safe_z(X[:, i]) for i in range(X.shape[1])])

phi_latent = PCA(n_components=1).fit_transform(X).flatten()
dat["phi_latent"] = phi_latent
dat["phi_clean"] = rank_gaussian(phi_latent)

dat["dphi"] = dat.groupby("label")["phi_clean"].diff()
dat["d2phi"] = dat.groupby("label")["phi_clean"].diff().diff()
dat["dh"] = dat.groupby("label")["h"].diff()
dat["da"] = dat.groupby("label")["a"].diff()
dat["deps"] = dat.groupby("label")["eps"].diff()
dat["ddpsi"] = dat.groupby("label")["dpsi"].diff()

out_csv = os.path.join(OUT_DIR, "new_phi_dataset.csv")
dat.to_csv(out_csv, index=False)

print("POINT_DIR =", POINT_DIR)
print("n_files =", len(files))
print("constructed rows =", len(dat))
print("n_sessions =", dat["label"].nunique())
print("Saved:", out_csv)

In [ ]:
# ============================================================
# Chapter 7 Analysis Script:
# Search, Fix, and Evaluate the Best Intersection Model
# for Observed Intersection Structure (Data Preparation for Figure 14)
# ------------------------------------------------------------
# Purpose:
# This script prepares the quantitative results and point-level data
# used for the Chapter 7 analysis of observed intersection structure,
# corresponding to the figure later summarized as Figure 14 in the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"  [oai_citation:1‡Intersection_Defined_Phase_Coordinates_Reveal_Localized_Selection_and_a_Non_Closed_Observational_Structure-2.pdf](sediment://file_0000000035b0720bbaa688f92dfff37e)
#
# The goal is to identify, on the training split only, the best
# intersection-variable construction that most strongly exceeds the
# quantum-only baseline, and then evaluate that fixed model on the
# independent test split.
#
# The script also compares the selected best model against:
#   - neural-only model
#   - quantum-only model
#   - simple mean model
# and further tests whether the observed switching structure:
#   - survives comparison with single-system baselines
#   - collapses under temporal misalignment
#   - exceeds a block-permuted null distribution
# ============================================================



import os
import numpy as np
import pandas as pd
from itertools import combinations
from statistics import NormalDist

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.stats import binomtest

# =========================================
# CONFIG
# =========================================
BASE_DIR = "IDPC_Reproduction"
IN_CSV = os.path.join(BASE_DIR, "Chapter7", "new_phi_dataset.csv")
OUT_DIR = os.path.join(BASE_DIR, "Chapter7")
os.makedirs(OUT_DIR, exist_ok=True)

SESSION_COL = "label"
INDEX_COL = "idx_in_session"


REF_COLS = ["h", "dh", "da", "deps"]
REF_MODE = "pca"

NEURAL_ONLY_COLS = ["h", "dh", "da"]
NEURAL_ONLY_MODE = "pca"

QUANTUM_ONLY_COLS = ["deps", "dpsi"]
QUANTUM_ONLY_MODE = "pca"

SIMPLE_MEAN_COLS = ["h", "dh", "da", "deps"]
SIMPLE_MEAN_MODE = "mean"

SEARCH_COLS = ["h", "a", "dh", "da", "deps", "dpsi"]
SEARCH_K_LIST = [2, 3, 4]
SEARCH_MODE_LIST = ["pca", "mean", "weighted"]

N_PERM = 200
BLOCK = 12
SHIFT_LIST = [0, 2, 5, 10, 20]


KMEANS_RANDOM_STATE = 0
KMEANS_N_INIT = 20


# =========================================
# LOAD
# =========================================
df = pd.read_csv(IN_CSV)
df = df.sort_values([SESSION_COL, INDEX_COL]).reset_index(drop=True)

print("shape:", df.shape)
print("columns:", df.columns.tolist())

labels = sorted(df[SESSION_COL].unique())
train_labels = labels[:13]
test_labels = labels[13:]

df_train = df[df[SESSION_COL].isin(train_labels)].copy()
df_test = df[df[SESSION_COL].isin(test_labels)].copy()

print("train sessions:", len(train_labels))
print("test sessions :", len(test_labels))


# =========================================
# HELPERS
# =========================================
def zscore(X):
    X = np.asarray(X, float)
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd[sd == 0] = 1.0
    return (X - mu) / sd


def gaussianize(x):
    x = np.asarray(x, float)
    m = np.isfinite(x)
    out = np.full_like(x, np.nan, dtype=float)
    if m.sum() < 3:
        return out
    ranks = pd.Series(x[m]).rank(method="average").to_numpy() / (m.sum() + 1.0)
    nd = NormalDist()
    out[m] = np.array([nd.inv_cdf(v) for v in ranks], dtype=float)
    return out


def build_phi_candidate(sub, cols, mode):
    cols = list(cols)
    X = sub[cols].to_numpy(float)
    good = np.all(np.isfinite(X), axis=1)

    if good.sum() < 50:
        return None

    X = X[good]
    idx = sub.loc[good, INDEX_COL].to_numpy()

    Xz = zscore(X)

    if mode == "pca":
        phi_raw = PCA(n_components=1, svd_solver="full").fit_transform(Xz).ravel()
    elif mode == "mean":
        phi_raw = Xz.mean(axis=1)
    elif mode == "weighted":
        w = np.linspace(1.0, 2.0, Xz.shape[1])
        phi_raw = (Xz * w).sum(axis=1)
    else:
        raise ValueError(f"unknown mode: {mode}")

    phi = gaussianize(phi_raw)

    out = pd.DataFrame({
        INDEX_COL: idx,
        "phi": phi
    }).sort_values(INDEX_COL).reset_index(drop=True)

    out["dphi"] = np.gradient(out["phi"].to_numpy(float))
    return out


def mark_switch_neighborhood(winner, radius=2):
    winner = np.asarray(winner)
    out = np.zeros(len(winner), dtype=int)
    sw = np.where(np.diff(winner) != 0)[0] + 1
    for s in sw:
        lo = max(0, s - radius)
        hi = min(len(winner), s + radius + 1)
        out[lo:hi] = 1
    return out


def build_deltaC(phi):
    phi = np.asarray(phi, float)
    centers = np.quantile(phi, np.linspace(0.1, 0.9, 5))
    sigma = np.std(phi) * 0.6
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    out = []
    for v in phi:
        C = -((v - centers) ** 2) / (2 * sigma ** 2)
        s = np.sort(C)
        out.append(s[-1] - s[-2])
    return np.array(out, dtype=float)


def score_model_details(phi_df, centers_z, scaler):
    X = phi_df[["phi", "dphi"]].to_numpy(float)
    Xz = scaler.transform(X)

    rows = []
    for i in range(len(X)):
        d2 = np.sum((Xz[i] - centers_z) ** 2, axis=1)
        s = -d2
        o = np.argsort(s)[::-1]

        rows.append({
            "winner": int(o[0]),
            "sharp": float(s[o[0]] - s[o[1]]),
            "phi": float(X[i, 0]),
            "dphi": float(X[i, 1]),
        })

    d = pd.DataFrame(rows)
    d["near"] = mark_switch_neighborhood(d["winner"].to_numpy(), radius=2)
    d["deltaC"] = build_deltaC(d["phi"].to_numpy(float))

    sw = d[d["near"] == 1]
    non = d[d["near"] == 0]

    switch_gain = np.nan
    if len(sw) > 0 and len(non) > 0:
        switch_gain = float(sw["sharp"].mean() - non["sharp"].mean())

    deltaC_gain = np.nan
    if len(sw) > 0 and len(non) > 0:
        deltaC_gain = float(sw["deltaC"].mean() - non["deltaC"].mean())

    return {
        "switch_gain": switch_gain,
        "deltaC_gain": deltaC_gain,
        "sharp_mean": float(d["sharp"].mean()),
        "n_total": int(len(d)),
        "n_switch_near": int((d["near"] == 1).sum()),
    }, d


def evaluate_model_on_split(df_split, cols, mode, centers_z, scaler):
    rows = []
    scored_parts = []

    for label, sub in df_split.groupby(SESSION_COL):
        phi_df = build_phi_candidate(sub, cols, mode)
        if phi_df is None:
            continue

        metrics, scored = score_model_details(phi_df, centers_z, scaler)
        row = {"label": label, **metrics}
        rows.append(row)

        scored = scored.copy()
        scored["label"] = label
        scored_parts.append(scored)

    met_df = pd.DataFrame(rows)
    scored_df = pd.concat(scored_parts, ignore_index=True) if scored_parts else pd.DataFrame()
    return met_df, scored_df


def sign_test_p(diff_series):
    x = pd.Series(diff_series).dropna()
    pos = int((x > 0).sum())
    n = int((x != 0).sum())
    if n == 0:
        return np.nan
    return float(binomtest(pos, n=n, p=0.5, alternative="greater").pvalue)


def block_permute(arr, block=12, rng=None):
    arr = np.asarray(arr)
    n = len(arr)
    if rng is None:
        rng = np.random.default_rng(0)

    blocks = [arr[i:i+block] for i in range(0, n, block)]
    order = rng.permutation(len(blocks))
    out = np.concatenate([blocks[i] for i in order])
    return out[:n]


def deps_block_permuted_df(df_source, block=12, seed=0):
    rng = np.random.default_rng(seed)
    out_parts = []

    for _, sub in df_source.groupby(SESSION_COL):
        tmp = sub.copy()
        tmp["deps"] = block_permute(tmp["deps"].to_numpy(), block=block, rng=rng)
        out_parts.append(tmp)

    return pd.concat(out_parts, ignore_index=True)


def emp_p_ge(null_vals, obs):
    null_vals = np.asarray(null_vals, float)
    return float((np.sum(null_vals >= obs) + 1) / (len(null_vals) + 1))


# =========================================
# FIXED REFERENCE FROM TRAIN
# =========================================
train_ref_parts = []
for _, sub in df_train.groupby(SESSION_COL):
    p = build_phi_candidate(sub, REF_COLS, REF_MODE)
    if p is not None:
        train_ref_parts.append(p)

train_ref_phi = pd.concat(train_ref_parts, ignore_index=True)

X_ref = train_ref_phi[["phi", "dphi"]].to_numpy(float)
scaler = StandardScaler().fit(X_ref)
Xz_ref = scaler.transform(X_ref)

kmeans = KMeans(
    n_clusters=5,
    n_init=KMEANS_N_INIT,
    random_state=KMEANS_RANDOM_STATE
).fit(Xz_ref)
centers_z = kmeans.cluster_centers_


# =========================================
# FIX QUANTUM-ONLY BASELINE ON TRAIN
# =========================================
quantum_train_df, _ = evaluate_model_on_split(
    df_train, QUANTUM_ONLY_COLS, QUANTUM_ONLY_MODE, centers_z, scaler
)
quantum_train_map = dict(zip(quantum_train_df["label"], quantum_train_df["switch_gain"]))


# =========================================
# SEARCH ON TRAIN ONLY
# objective:
# =========================================
search_rows = []

for k in SEARCH_K_LIST:
    for cols in combinations(SEARCH_COLS, k):
        for mode in SEARCH_MODE_LIST:
            cand_train_df, _ = evaluate_model_on_split(
                df_train, cols, mode, centers_z, scaler
            )

            if cand_train_df.empty:
                continue

            merged = cand_train_df[["label", "switch_gain"]].merge(
                quantum_train_df[["label", "switch_gain"]],
                on="label",
                suffixes=("_cand", "_q")
            )

            if merged.empty:
                continue

            d = merged["switch_gain_cand"] - merged["switch_gain_q"]

            mean_adv = float(d.mean())
            median_adv = float(d.median())
            p_adv = sign_test_p(d)

            search_rows.append({
                "cols": str(tuple(cols)),
                "mode": mode,
                "mean_adv_vs_quantum_train": mean_adv,
                "median_adv_vs_quantum_train": median_adv,
                "sign_test_p_vs_quantum_train": p_adv,
                "cand_train_mean_switch_gain": float(merged["switch_gain_cand"].mean()),
                "quantum_train_mean_switch_gain": float(merged["switch_gain_q"].mean()),
                "n_sessions": int(len(merged)),
            })

search_df = pd.DataFrame(search_rows)

search_df = search_df.sort_values(
    [
        "mean_adv_vs_quantum_train",
        "median_adv_vs_quantum_train",
        "sign_test_p_vs_quantum_train",
        "cand_train_mean_switch_gain",
    ],
    ascending=[False, False, True, False]
).reset_index(drop=True)

print("\n=== TRUE SEARCH RESULT (TRAIN ONLY, objective = beat quantum_only) ===\n")
print(search_df.head(20).to_string(index=False))

search_df.to_csv(os.path.join(OUT_DIR, "true_search_train_only_vs_quantum.csv"), index=False)

best_search_row = search_df.iloc[0].copy()
BEST_TRUE_COLS = list(eval(best_search_row["cols"]))
BEST_TRUE_MODE = str(best_search_row["mode"])

print("\n=== SELECTED TRUE BEST ===\n")
print("BEST_TRUE_COLS =", BEST_TRUE_COLS)
print("BEST_TRUE_MODE =", BEST_TRUE_MODE)
print("TRAIN mean_adv_vs_quantum =", float(best_search_row["mean_adv_vs_quantum_train"]))
print("TRAIN sign_test_p_vs_quantum =", float(best_search_row["sign_test_p_vs_quantum_train"]))


# =========================================
# EVALUATE FIXED BEST ON TEST ONLY
# =========================================
best_test_df, best_scored_test = evaluate_model_on_split(
    df_test, BEST_TRUE_COLS, BEST_TRUE_MODE, centers_z, scaler
)
best_test_df["model"] = "best_true_search"

neural_test_df, _ = evaluate_model_on_split(
    df_test, NEURAL_ONLY_COLS, NEURAL_ONLY_MODE, centers_z, scaler
)
neural_test_df["model"] = "neural_only_pca"

quantum_test_df, _ = evaluate_model_on_split(
    df_test, QUANTUM_ONLY_COLS, QUANTUM_ONLY_MODE, centers_z, scaler
)
quantum_test_df["model"] = "quantum_only_pca"

simple_test_df, _ = evaluate_model_on_split(
    df_test, SIMPLE_MEAN_COLS, SIMPLE_MEAN_MODE, centers_z, scaler
)
simple_test_df["model"] = "simple_mean_best"

all_test_metrics = pd.concat(
    [best_test_df, neural_test_df, quantum_test_df, simple_test_df],
    ignore_index=True
)

print("\n=== SESSION-WISE TEST METRICS ===\n")
print(all_test_metrics.to_string(index=False))

test_summary = (
    all_test_metrics.groupby("model")[["switch_gain", "deltaC_gain", "sharp_mean"]]
    .agg(["mean", "std", "min", "max"])
)
print("\n=== TEST SUMMARY ===\n")
print(test_summary.to_string())


# =========================================
# PAIRED DIFFERENCES ON TEST
# =========================================
paired_rows = []
for label in sorted(all_test_metrics["label"].unique()):
    row_best = all_test_metrics[
        (all_test_metrics["label"] == label) &
        (all_test_metrics["model"] == "best_true_search")
    ].iloc[0]

    for other in ["neural_only_pca", "quantum_only_pca", "simple_mean_best"]:
        row_o = all_test_metrics[
            (all_test_metrics["label"] == label) &
            (all_test_metrics["model"] == other)
        ].iloc[0]

        paired_rows.append({
            "label": label,
            "comparison": f"best_minus_{other}",
            "d_switch_gain": float(row_best["switch_gain"] - row_o["switch_gain"]),
            "d_deltaC_gain": float(row_best["deltaC_gain"] - row_o["deltaC_gain"]),
        })

paired_df = pd.DataFrame(paired_rows)

print("\n=== PAIRED DIFFERENCES VS BEST ===\n")
print(
    paired_df.groupby("comparison")[["d_switch_gain", "d_deltaC_gain"]]
    .agg(["mean", "std", "min", "max"])
    .to_string()
)

print("\n=== SIGN-TEST P VALUES (best > other) ===")
for comp, sub in paired_df.groupby("comparison"):
    print(comp)
    print("  switch_gain p =", sign_test_p(sub["d_switch_gain"]))
    print("  deltaC_gain p =", sign_test_p(sub["d_deltaC_gain"]))


# =========================================
# BLOCK PERMUTATION TEST ON TEST
# =========================================
perm_rows = []

for i in range(N_PERM):
    df_perm = deps_block_permuted_df(df_test, block=BLOCK, seed=i)
    met_perm, _ = evaluate_model_on_split(
        df_perm, BEST_TRUE_COLS, BEST_TRUE_MODE, centers_z, scaler
    )

    perm_rows.append({
        "iter": i,
        "switch_gain": float(met_perm["switch_gain"].mean()),
        "deltaC_gain": float(met_perm["deltaC_gain"].mean()),
    })

perm_df = pd.DataFrame(perm_rows)

obs_best_switch = float(best_test_df["switch_gain"].mean())
obs_best_deltaC = float(best_test_df["deltaC_gain"].mean())

perm_p_switch = emp_p_ge(perm_df["switch_gain"], obs_best_switch)
perm_p_deltaC = emp_p_ge(perm_df["deltaC_gain"], obs_best_deltaC)

print("\n=== BLOCK PERMUTATION SUMMARY ===\n")
print(perm_df.describe().to_string())

print("\n=== EMPIRICAL PERMUTATION P VALUES ===")
print("switch_gain p =", perm_p_switch)
print("deltaC_gain p =", perm_p_deltaC)


# =========================================
# TEMPORAL SHIFT TEST ON TEST
# =========================================
shift_rows = []

for shift in SHIFT_LIST:
    parts = []

    for _, sub in df_test.groupby(SESSION_COL):
        tmp = sub.copy()
        if shift != 0:
            tmp["deps"] = tmp["deps"].shift(shift)
        parts.append(tmp)

    df_shift = pd.concat(parts, ignore_index=True)
    met_shift, _ = evaluate_model_on_split(
        df_shift, BEST_TRUE_COLS, BEST_TRUE_MODE, centers_z, scaler
    )

    shift_rows.append({
        "shift": shift,
        "switch_gain": float(met_shift["switch_gain"].mean()),
        "deltaC_gain": float(met_shift["deltaC_gain"].mean()),
    })

shift_df = pd.DataFrame(shift_rows)

print("\n=== TEMPORAL SHIFT TEST ===\n")
print(shift_df.to_string(index=False))


# =========================================
# FINAL CLAIM TABLE
# =========================================
mean_best = float(best_test_df["switch_gain"].mean())
mean_neural = float(neural_test_df["switch_gain"].mean())
mean_quantum = float(quantum_test_df["switch_gain"].mean())
mean_simple = float(simple_test_df["switch_gain"].mean())

p_best_vs_neural = sign_test_p(
    paired_df.loc[paired_df["comparison"] == "best_minus_neural_only_pca", "d_switch_gain"]
)
p_best_vs_quantum = sign_test_p(
    paired_df.loc[paired_df["comparison"] == "best_minus_quantum_only_pca", "d_switch_gain"]
)
p_best_vs_simple = sign_test_p(
    paired_df.loc[paired_df["comparison"] == "best_minus_simple_mean_best", "d_switch_gain"]
)

final_claim_table = pd.DataFrame([
    {
        "claim": "Best exceeds neural-only on switch_gain",
        "evidence": (
            f"best_true_search cols={BEST_TRUE_COLS}, mode={BEST_TRUE_MODE}; "
            f"best switch_gain={mean_best:.6f}, neural-only={mean_neural:.6f}; "
            f"paired sign-test p={p_best_vs_neural:.6f}"
        )
    },
    {
        "claim": "Best compared with quantum-only on switch_gain",
        "evidence": (
            f"best_true_search cols={BEST_TRUE_COLS}, mode={BEST_TRUE_MODE}; "
            f"best switch_gain={mean_best:.6f}, quantum-only={mean_quantum:.6f}; "
            f"paired sign-test p={p_best_vs_quantum:.6f}"
        )
    },
    {
        "claim": "Best exceeds simple mean on switch_gain",
        "evidence": (
            f"best_true_search cols={BEST_TRUE_COLS}, mode={BEST_TRUE_MODE}; "
            f"best switch_gain={mean_best:.6f}, simple-mean={mean_simple:.6f}; "
            f"paired sign-test p={p_best_vs_simple:.6f}"
        )
    },
    {
        "claim": "Best exceeds block-permuted null on switch_gain",
        "evidence": (
            f"best_true_search cols={BEST_TRUE_COLS}, mode={BEST_TRUE_MODE}; "
            f"obs switch_gain={obs_best_switch:.6f}, null mean={perm_df['switch_gain'].mean():.6f}, "
            f"empirical p={perm_p_switch:.6f}"
        )
    },
    {
        "claim": "Best degrades under temporal misalignment",
        "evidence": (
            f"best_true_search cols={BEST_TRUE_COLS}, mode={BEST_TRUE_MODE}; "
            f"shift0={shift_df.loc[shift_df['shift']==0, 'switch_gain'].iloc[0]:.6f}, "
            f"shift20={shift_df.loc[shift_df['shift']==20, 'switch_gain'].iloc[0]:.6f}"
        )
    }
])

print("\n=== FINAL CLAIM TABLE ===\n")
print(final_claim_table.to_string(index=False))


# =========================================
# SAVE
# =========================================
best_scored_test.to_csv(os.path.join(OUT_DIR, "best_true_search_scored_points.csv"), index=False)
all_test_metrics.to_csv(os.path.join(OUT_DIR, "session_wise_metrics_all_models.csv"), index=False)
paired_df.to_csv(os.path.join(OUT_DIR, "paired_differences_vs_best.csv"), index=False)
perm_df.to_csv(os.path.join(OUT_DIR, "block_permutation_test.csv"), index=False)
shift_df.to_csv(os.path.join(OUT_DIR, "temporal_shift_test.csv"), index=False)
final_claim_table.to_csv(os.path.join(OUT_DIR, "final_claim_table.csv"), index=False)

print("\nSaved:")
print(os.path.join(OUT_DIR, "true_search_train_only_vs_quantum.csv"))
print(os.path.join(OUT_DIR, "best_true_search_scored_points.csv"))
print(os.path.join(OUT_DIR, "session_wise_metrics_all_models.csv"))
print(os.path.join(OUT_DIR, "paired_differences_vs_best.csv"))
print(os.path.join(OUT_DIR, "block_permutation_test.csv"))
print(os.path.join(OUT_DIR, "temporal_shift_test.csv"))
print(os.path.join(OUT_DIR, "final_claim_table.csv"))

In [ ]:
# ============================================================
# Chapter 7 Figure Script:
# Figure 14 — Observed Intersection Structure
# ------------------------------------------------------------
# Purpose:
# This script generates Figure 14 for Chapter 7 of the paper:
# "Intersection-Defined Phase Coordinates Reveal Localized Selection
#  and a Non-Closed Observational Structure"
#
# The figure visualizes the empirical structure of the best
# intersection variable selected in the separate train-only search.
#
# Within the framework of the paper, the observed structure is
# interpreted as an empirical trace of O3:
# a residual signature of a non-internal observational constraint
# that manifests on the intersection-defined phase space.
#
# Importantly, O3 is not introduced here as a directly observable
# entity, but as an operational construct inferred from:
#   - structured cross-system agreement
#   - its dependence on temporal alignment
#   - its degradation under controlled perturbations
#
# Figure 14 summarizes four core results:
#   A. localized switching structure on the (φ, dφ) phase space
#   B. superiority of the best intersection model over
#      single-system baselines
#   C. weakening of the structure under temporal misalignment
#   D. separation from a block-permuted null distribution
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# PATH
# =========================================================
BASE_DIR = "IDPC_Reproduction"
FIG_DIR = os.path.join(BASE_DIR, "Chapter7")

BEST_SCATTER = os.path.join(FIG_DIR, "best_true_search_scored_points.csv")
MODEL_SUMMARY = os.path.join(FIG_DIR, "session_wise_metrics_all_models.csv")
SHIFT_CSV = os.path.join(FIG_DIR, "temporal_shift_test.csv")
PERM_CSV = os.path.join(FIG_DIR, "block_permutation_test.csv")

OUT_PNG = os.path.join(FIG_DIR, "Chapter7_Fig_Observed_Intersection_Structure.png")

# =========================================================
# LOAD
# =========================================================
best_scored = pd.read_csv(BEST_SCATTER)
model_metrics = pd.read_csv(MODEL_SUMMARY)
shift_df = pd.read_csv(SHIFT_CSV)
perm_df = pd.read_csv(PERM_CSV)

# =========================================================
# PANEL B DATA
# =========================================================
model_order = [
    "best_true_search",
    "neural_only_pca",
    "quantum_only_pca",
    "simple_mean_best",
]

label_map = {
    "best_true_search": "best φ",
    "neural_only_pca": "neural only",
    "quantum_only_pca": "quantum only",
    "simple_mean_best": "simple mean",
}

bar_vals = (
    model_metrics.groupby("model", sort=False)["switch_gain"]
    .mean()
    .reindex(model_order)
)

# =========================================================
# PANEL C DATA
# =========================================================
shift_df = shift_df.sort_values("shift").reset_index(drop=True)

# =========================================================
# PANEL D DATA
# =========================================================
observed_switch_gain = float(bar_vals.loc["best_true_search"])
null_mean = float(perm_df["switch_gain"].mean())
perm_p = float((np.sum(perm_df["switch_gain"].to_numpy() >= observed_switch_gain) + 1) / (len(perm_df) + 1))

# =========================================================
# PLOT
# =========================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# -------------------------
# A. Localized switching structure
# -------------------------
ax = axes[0, 0]

x = best_scored["phi"].to_numpy(float)
y = best_scored["dphi"].to_numpy(float)
z = best_scored["near"].to_numpy(float)

hb = ax.hexbin(
    x,
    y,
    C=z,
    reduce_C_function=np.mean,
    gridsize=55,
    mincnt=5,
    vmin=0.0,
    vmax=1.0
)

cbar = fig.colorbar(hb, ax=ax)
cbar.set_label("switch rate (best φ)")

ax.set_title("A. Localized switching structure of the best intersection variable")
ax.set_xlabel(r"$\phi$")
ax.set_ylabel(r"$d\phi$")

# -------------------------
# B. Bar chart
# -------------------------
ax = axes[0, 1]

x_labels = [label_map[m] for m in model_order]
y_vals = [float(bar_vals.loc[m]) for m in model_order]

bars = ax.bar(x_labels, y_vals)

for bar, val in zip(bars, y_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.01,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=12,
    )

ax.set_ylabel("switch_gain")
ax.set_title("B. Best φ exceeds single-system models")
ax.set_ylim(0, max(y_vals) + 0.12)
ax.tick_params(axis="x", rotation=20)

# -------------------------
# C. Temporal shift
# -------------------------
ax = axes[1, 0]

ax.plot(
    shift_df["shift"],
    shift_df["switch_gain"],
    marker="o"
)

for _, row in shift_df.iterrows():
    ax.text(
        float(row["shift"]),
        float(row["switch_gain"]) + 0.004,
        f"{float(row['switch_gain']):.3f}",
        ha="center",
        va="bottom",
        fontsize=11,
    )

ax.set_title("C. Temporal misalignment weakens switching")
ax.set_xlabel("temporal shift")
ax.set_ylabel("switch_gain")

# -------------------------
# D. Permutation histogram
# -------------------------
ax = axes[1, 1]

ax.hist(perm_df["switch_gain"], bins=30, alpha=0.8)
ax.axvline(observed_switch_gain, linewidth=2)

annot = (
    f"observed = {observed_switch_gain:.3f}\n"
    f"null mean = {null_mean:.3f}\n"
    f"p = {perm_p:.3f}"
)
ax.text(
    0.03,
    0.97,
    annot,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=14,
)

ax.set_title("D. Observed switching exceeds permuted null")
ax.set_xlabel("switch_gain")
ax.set_ylabel("count")

# -------------------------
# SAVE
# -------------------------
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PNG)